# **Metriche Matomo - DSpace**

## **Setup**

### Install requirements

Installare le dipendenze dal terminale con `pip install -r requirements.txt` prima dell esecuzione batch.

### Import packages and configuration loading

In [ ]:
# =========================================================
# IMPORT PACKAGES AND CONFIGURATION LOADING
# =========================================================

import os
import requests
import pandas as pd
from pathlib import Path
import yaml

EXPORT_SOURCE = "matomo"
MONTHLY_EXPORT_FILES = {
    "matomo_actions_per_visit_reference_month.csv",
    "matomo_average_visit_duration_reference_month.csv",
    "matomo_bounce_rate_reference_month.csv",
    "matomo_campaign_visits_reference_month.csv",
    "matomo_direct_visits_reference_month_details.csv",
    "matomo_pageviews_reference_month_action_details.csv",
    "matomo_top_referrer_websites_reference_month.csv",
    "matomo_top_visited_pages_reference_month.csv",
    "matomo_typology_of_page_views_reference_month.csv",
    "matomo_visits_by_browser_reference_month.csv",
    "matomo_visits_by_city_reference_month.csv",
    "matomo_visits_by_continent_reference_month.csv",
    "matomo_visits_by_country_reference_month.csv",
    "matomo_visits_by_device_type_reference_month.csv",
    "matomo_visits_by_operating_system_reference_month.csv",
    "matomo_visits_from_search_engines_reference_month.csv",
    "matomo_visits_from_social_networks_reference_month.csv",
    "matomo_visits_from_websites_reference_month.csv",
    "matomo_visits_reference_month.csv",
}


def load_config(config_path=None):
    config_path = config_path or os.environ.get(
        "DSPACE_REPORTING_CONFIG",
        "../config/settings.local.yaml",
    )
    with open(config_path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def get_scoped_export_dir(scope):
    export_root = PROJECT_ROOT / config["paths"].get("exports_root", "exports")
    base_dir = export_root / EXPORT_SOURCE / scope
    if config.get("run", {}).get("create_month_subfolders", True):
        return base_dir / REPORT_MONTH
    return base_dir


def export_path(filename):
    scope = "monthly" if filename in MONTHLY_EXPORT_FILES else "always"
    output_path = get_scoped_export_dir(scope) / filename
    if EXPORT_OUTPUTS:
        output_path.parent.mkdir(parents=True, exist_ok=True)
    return output_path


def write_csv(dataframe, output_path, *args, **kwargs):
    if EXPORT_OUTPUTS:
        dataframe.to_csv(output_path, *args, **kwargs)
    else:
        print(f"Display-only mode, export skipped: {output_path.resolve()}")


config = load_config()

REPORT_MONTH = config["run"]["reference_month"]          # es. 2026-04
REPORT_MONTH_DATE = f"{REPORT_MONTH}-01"                 # es. 2026-04-01
EXPORT_OUTPUTS = config.get("run", {}).get("export_outputs", True)
ENVIRONMENT = config["project"]["environment"]
TIMEZONE = config["project"]["timezone"]

PROJECT_ROOT = Path.cwd().parent
EXPORT_ROOT = PROJECT_ROOT / config["paths"].get("exports_root", "exports")
source_export_dir = EXPORT_ROOT / EXPORT_SOURCE
monthly_export_dir = get_scoped_export_dir("monthly")
always_export_dir = get_scoped_export_dir("always")
export_dir = source_export_dir

MATOMO_BASE_URL = config["matomo"]["base_url"].rstrip("/")
MATOMO_SITE_ID = config["matomo"]["site_id"]
MATOMO_TOKEN_AUTH = config["matomo"]["token_auth"]
MATOMO_DOMAIN_FILTER = config["matomo"].get("domain_filter")
MATOMO_VERIFY_SSL = config["matomo"].get("verify_ssl", True)
MATOMO_TIMEOUT = config["matomo"].get("timeout", 30)

MATOMO_API_URL = f"{MATOMO_BASE_URL}/index.php"

print("=== Matomo context of implementation ===")
print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Environment: {ENVIRONMENT}")
print(f"Timezone: {TIMEZONE}")
print(f"Reference month: {REPORT_MONTH}")
print(f"Monthly export directory: {monthly_export_dir.resolve()}")
print(f"Always export directory: {always_export_dir.resolve()}")
print(f"Matomo base URL: {MATOMO_BASE_URL}")
print(f"Matomo site ID: {MATOMO_SITE_ID}")
print(f"Matomo domain filter: {MATOMO_DOMAIN_FILTER}")
print(f"Matomo SSL verification: {MATOMO_VERIFY_SSL}")
print()


### Matomo API connection check

In [ ]:
# =========================================================
# MATOMO API CONNECTION CHECK
# =========================================================

def matomo_api_call(method, params=None):
    """
    Calls the Matomo Reporting API.

    The token is sent via POST parameters and is never printed.
    """

    if params is None:
        params = {}

    payload = {
        "module": "API",
        "method": method,
        "format": "JSON",
        "idSite": MATOMO_SITE_ID,
        "token_auth": MATOMO_TOKEN_AUTH,
        **params
    }

    response = requests.post(
        MATOMO_API_URL,
        data=payload,
        timeout=MATOMO_TIMEOUT,
        verify=MATOMO_VERIFY_SSL
    )

    response.raise_for_status()

    data = response.json()

    if isinstance(data, dict) and data.get("result") == "error":
        raise RuntimeError(
            f"Matomo API error for method {method}: {data.get('message')}"
        )

    return data


print("=== Matomo connection check ===")

try:
    matomo_version = matomo_api_call(
        "API.getMatomoVersion",
        params={}
    )

    print("Matomo API reachable.")
    print(f"Matomo version: {matomo_version}")

except Exception as e:
    print("Matomo API connection failed.")
    print(str(e))
    raise


try:
    site_info = matomo_api_call(
        "SitesManager.getSiteFromId",
        params={}
    )

    print()
    print("Matomo site found.")
    print(f"Site ID: {MATOMO_SITE_ID}")

    if isinstance(site_info, dict):
        print(f"Site name: {site_info.get('name')}")
        print(f"Main URL: {site_info.get('main_url')}")

except Exception as e:
    print("Matomo site check failed.")
    print(str(e))
    raise


try:
    visits_summary = matomo_api_call(
        "VisitsSummary.get",
        params={
            "period": "month",
            "date": REPORT_MONTH_DATE
        }
    )

    print()
    print("VisitsSummary check successful.")
    print(f"Reference month: {REPORT_MONTH}")
    print(f"Visits: {visits_summary.get('nb_visits')}")
    print(f"Unique visitors: {visits_summary.get('nb_uniq_visitors')}")
    print(f"Actions: {visits_summary.get('nb_actions')}")
    print(f"Bounce count: {visits_summary.get('bounce_count')}")
    print(f"Bounce rate: {visits_summary.get('bounce_rate')}")
    print(f"Actions per visit: {visits_summary.get('nb_actions_per_visit')}")
    print(f"Average time on site: {visits_summary.get('avg_time_on_site')}")

except Exception as e:
    print("VisitsSummary check failed.")
    print(str(e))
    raise

## **General Traffic Overview**

### Total visits

In [ ]:
# =========================================================
# TOTAL VISITS
# =========================================================

# - metric_name
# - visits_count
# - period
# - date_range
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_total_visits = matomo_api_call(
    "VisitsSummary.get",
    params={
        "period": "range",
        "date": "2000-01-01,today"
    }
)

total_visits = response_total_visits.get("nb_visits", 0)

df_total_visits = pd.DataFrame([
    {
        "metric_name": "total_visits",
        "visits_count": total_visits,
        "period": "range",
        "date_range": "2000-01-01,today",
        "site_id": MATOMO_SITE_ID
    }
])

df_total_visits["reference_month"] = REPORT_MONTH
df_total_visits["environment"] = ENVIRONMENT
df_total_visits["source"] = "matomo"
df_total_visits["metric_definition"] = (
    "total number of visits recorded by Matomo for the tracked DSpace site "
    "over the full available reporting range"
)

output_total_visits = export_path("matomo_total_visits.csv")

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_total_visits, output_total_visits, index=False)

print(f"Total visits: {total_visits}")

display(df_total_visits)

print(f"File saved to: {output_total_visits.resolve()}")

### Unique visitors by month

In [ ]:
# =========================================================
# UNIQUE VISITORS BY MONTH
# =========================================================

# - month
# - unique_visitors_count
# - visits_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_unique_visitors_by_month = matomo_api_call(
    "VisitsSummary.get",
    params={
        "period": "month",
        "date": "last12"
    }
)


def normalize_matomo_period_values(values):
    """
    Matomo may return period values either as a dictionary or as a list
    containing one dictionary. This helper normalizes both cases.
    """

    if isinstance(values, dict):
        return values

    if isinstance(values, list):
        if len(values) > 0 and isinstance(values[0], dict):
            return values[0]

    return {}


unique_visitors_by_month_rows = []

for month, values in response_unique_visitors_by_month.items():
    normalized_values = normalize_matomo_period_values(values)

    unique_visitors_by_month_rows.append({
        "month": month,
        "unique_visitors_count": normalized_values.get("nb_uniq_visitors", 0),
        "visits_count": normalized_values.get("nb_visits", 0),
        "actions_count": normalized_values.get("nb_actions", 0),
        "bounce_count": normalized_values.get("bounce_count", 0),
        "bounce_rate": normalized_values.get("bounce_rate"),
        "actions_per_visit": normalized_values.get("nb_actions_per_visit"),
        "average_time_on_site": normalized_values.get("avg_time_on_site"),
        "period": "month",
        "site_id": MATOMO_SITE_ID
    })

df_unique_visitors_by_month = pd.DataFrame(unique_visitors_by_month_rows)

df_unique_visitors_by_month["reference_month"] = REPORT_MONTH
df_unique_visitors_by_month["environment"] = ENVIRONMENT
df_unique_visitors_by_month["source"] = "matomo"
df_unique_visitors_by_month["metric_definition"] = (
    "monthly number of unique visitors recorded by Matomo for the tracked DSpace site, "
    "together with visits, actions, bounce count, bounce rate, actions per visit "
    "and average time on site"
)

output_unique_visitors_by_month = (
    export_path("matomo_unique_visitors_by_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_unique_visitors_by_month, 
    output_unique_visitors_by_month,
    index=False
)

print(f"Monthly unique visitor rows extracted: {len(df_unique_visitors_by_month)}")

display(
    df_unique_visitors_by_month
    .sort_values("month")
)

print(f"File saved to: {output_unique_visitors_by_month.resolve()}")

### Total actions

In [ ]:
# =========================================================
# TOTAL ACTIONS
# =========================================================

# - metric_name
# - actions_count
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_total_actions = matomo_api_call(
    "VisitsSummary.get",
    params={
        "period": "range",
        "date": "2000-01-01,today"
    }
)

total_actions = response_total_actions.get("nb_actions", 0)

df_total_actions = pd.DataFrame([
    {
        "metric_name": "total_actions",
        "actions_count": total_actions,
        "period": "range",
        "site_id": MATOMO_SITE_ID
    }
])

df_total_actions["reference_month"] = REPORT_MONTH
df_total_actions["environment"] = ENVIRONMENT
df_total_actions["source"] = "matomo"
df_total_actions["metric_definition"] = (
    "total number of actions recorded by Matomo for the tracked DSpace site. "
    "Actions may include pageviews, downloads, outlinks, site searches or events, "
    "depending on the Matomo tracking configuration."
)

output_total_actions = export_path("matomo_total_actions.csv")

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_total_actions, output_total_actions, index=False)

print(f"Total actions: {total_actions}")

display(df_total_actions)

print(f"File saved to: {output_total_actions.resolve()}")

### Average visit duration

In [ ]:
# =========================================================
# AVERAGE VISIT DURATION
# =========================================================

# - metric_name
# - average_visit_duration_seconds
# - average_visit_duration_minutes
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_average_visit_duration = matomo_api_call(
    "VisitsSummary.get",
    params={
        "period": "range",
        "date": "2000-01-01,today"
    }
)

average_visit_duration_seconds = response_average_visit_duration.get(
    "avg_time_on_site",
    0
)

average_visit_duration_minutes = (
    round(average_visit_duration_seconds / 60, 2)
    if average_visit_duration_seconds is not None
    else 0
)

df_average_visit_duration = pd.DataFrame([
    {
        "metric_name": "average_visit_duration",
        "average_visit_duration_seconds": average_visit_duration_seconds,
        "average_visit_duration_minutes": average_visit_duration_minutes,
        "period": "range",
        "site_id": MATOMO_SITE_ID
    }
])

df_average_visit_duration["reference_month"] = REPORT_MONTH
df_average_visit_duration["environment"] = ENVIRONMENT
df_average_visit_duration["source"] = "matomo"
df_average_visit_duration["metric_definition"] = (
    "average visit duration recorded by Matomo for the tracked DSpace site, "
    "expressed in seconds and minutes"
)

output_average_visit_duration = (
    export_path("matomo_average_visit_duration.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_average_visit_duration, 
    output_average_visit_duration,
    index=False
)

print(
    "Average visit duration: "
    f"{average_visit_duration_seconds} seconds "
    f"({average_visit_duration_minutes} minutes)"
)

display(df_average_visit_duration)

print(f"File saved to: {output_average_visit_duration.resolve()}")

### Bounce count

In [ ]:
# =========================================================
# BOUNCE COUNT
# =========================================================

# - metric_name
# - bounce_count
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_bounce_count = matomo_api_call(
    "VisitsSummary.get",
    params={
        "period": "range",
        "date": "2000-01-01,today"
    }
)

bounce_count = response_bounce_count.get("bounce_count", 0)

df_bounce_count = pd.DataFrame([
    {
        "metric_name": "bounce_count",
        "bounce_count": bounce_count,
        "period": "range",
        "site_id": MATOMO_SITE_ID
    }
])

df_bounce_count["reference_month"] = REPORT_MONTH
df_bounce_count["environment"] = ENVIRONMENT
df_bounce_count["source"] = "matomo"
df_bounce_count["metric_definition"] = (
    "number of bounced visits recorded by Matomo for the tracked DSpace site. "
    "A bounced visit is a visit with only one tracked action."
)

output_bounce_count = export_path("matomo_bounce_count.csv")

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_bounce_count, output_bounce_count, index=False)

print(f"Bounce count: {bounce_count}")

display(df_bounce_count)

print(f"File saved to: {output_bounce_count.resolve()}")

### Bounce rate

In [ ]:
# =========================================================
# BOUNCE RATE
# =========================================================

# - metric_name
# - bounce_rate
# - bounce_count
# - visits_count
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_bounce_rate = matomo_api_call(
    "VisitsSummary.get",
    params={
        "period": "range",
        "date": "2000-01-01,today"
    }
)

bounce_rate = response_bounce_rate.get("bounce_rate")
bounce_count = response_bounce_rate.get("bounce_count", 0)
visits_count = response_bounce_rate.get("nb_visits", 0)

df_bounce_rate = pd.DataFrame([
    {
        "metric_name": "bounce_rate",
        "bounce_rate": bounce_rate,
        "bounce_count": bounce_count,
        "visits_count": visits_count,
        "period": "range",
        "site_id": MATOMO_SITE_ID
    }
])

df_bounce_rate["reference_month"] = REPORT_MONTH
df_bounce_rate["environment"] = ENVIRONMENT
df_bounce_rate["source"] = "matomo"
df_bounce_rate["metric_definition"] = (
    "percentage of bounced visits recorded by Matomo for the tracked DSpace site. "
    "A bounced visit is a visit with only one tracked action."
)

output_bounce_rate = export_path("matomo_bounce_rate.csv")

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_bounce_rate, output_bounce_rate, index=False)

print(f"Bounce rate: {bounce_rate}")
print(f"Bounce count: {bounce_count}")
print(f"Total visits: {visits_count}")

display(df_bounce_rate)

print(f"File saved to: {output_bounce_rate.resolve()}")

### Actions per visit

In [ ]:
# =========================================================
# ACTIONS PER VISIT
# =========================================================

# - metric_name
# - actions_per_visit
# - actions_count
# - visits_count
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_actions_per_visit = matomo_api_call(
    "VisitsSummary.get",
    params={
        "period": "range",
        "date": "2000-01-01,today"
    }
)

actions_per_visit = response_actions_per_visit.get("nb_actions_per_visit")
actions_count = response_actions_per_visit.get("nb_actions", 0)
visits_count = response_actions_per_visit.get("nb_visits", 0)

df_actions_per_visit = pd.DataFrame([
    {
        "metric_name": "actions_per_visit",
        "actions_per_visit": actions_per_visit,
        "actions_count": actions_count,
        "visits_count": visits_count,
        "period": "range",
        "site_id": MATOMO_SITE_ID
    }
])

df_actions_per_visit["reference_month"] = REPORT_MONTH
df_actions_per_visit["environment"] = ENVIRONMENT
df_actions_per_visit["source"] = "matomo"
df_actions_per_visit["metric_definition"] = (
    "average number of tracked actions per visit recorded by Matomo for the tracked "
    "DSpace site. Actions may include pageviews, downloads, outlinks, site searches "
    "or events, depending on the Matomo tracking configuration."
)

output_actions_per_visit = export_path("matomo_actions_per_visit.csv")

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_actions_per_visit, output_actions_per_visit, index=False)

print(f"Actions per visit: {actions_per_visit}")
print(f"Total actions: {actions_count}")
print(f"Total visits: {visits_count}")

display(df_actions_per_visit)

print(f"File saved to: {output_actions_per_visit.resolve()}")

### Max actions in a visit

In [ ]:
# =========================================================
# MAX ACTIONS IN A VISIT
# =========================================================

# - metric_name
# - max_actions_in_visit
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_max_actions_in_visit = matomo_api_call(
    "VisitsSummary.get",
    params={
        "period": "range",
        "date": "2000-01-01,today"
    }
)

max_actions_in_visit = response_max_actions_in_visit.get("max_actions", 0)

df_max_actions_in_visit = pd.DataFrame([
    {
        "metric_name": "max_actions_in_visit",
        "max_actions_in_visit": max_actions_in_visit,
        "period": "range",
        "site_id": MATOMO_SITE_ID
    }
])

df_max_actions_in_visit["reference_month"] = REPORT_MONTH
df_max_actions_in_visit["environment"] = ENVIRONMENT
df_max_actions_in_visit["source"] = "matomo"
df_max_actions_in_visit["metric_definition"] = (
    "maximum number of tracked actions recorded by Matomo within a single visit "
    "for the tracked DSpace site. Actions may include pageviews, downloads, outlinks, "
    "site searches or events, depending on the Matomo tracking configuration."
)

output_max_actions_in_visit = export_path("matomo_max_actions_in_visit.csv")

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_max_actions_in_visit, output_max_actions_in_visit, index=False)

print(f"Max actions in a visit: {max_actions_in_visit}")

display(df_max_actions_in_visit)

print(f"File saved to: {output_max_actions_in_visit.resolve()}")

## **Reference month traffic overview**

### Total visits in the reference month

In [ ]:
# =========================================================
# VISITS LIST IN THE REFERENCE MONTH WITH SEARCH DETAILS
# =========================================================

# - visit_id
# - visitor_id
# - visit_first_action_time
# - visit_last_action_time
# - visit_duration_seconds
# - actions_count
# - pageviews_count
# - searches_count
# - searches_detected_count
# - search_queries_detected
# - search_filters_detected
# - search_pages_detected
# - search_urls_detected
# - downloads_count
# - outlinks_count
# - events_count
# - referrer_type
# - referrer_name
# - referrer_url
# - country
# - region
# - city
# - latitude
# - longitude
# - browser
# - browser_version
# - operating_system
# - device_type
# - device_brand
# - device_model
# - resolution
# - language
# - site_id
# - period
# - period_start
# - period_end
# - reference_month
# - environment
# - source
# - metric_definition

from urllib.parse import urlparse, parse_qsl

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")


def get_matomo_visit_details(date_range, period="range", page_size=500):
    """
    Retrieves visit-level details from Matomo using Live.getLastVisitsDetails.

    Matomo returns visits in pages using filter_limit and filter_offset.
    """

    all_visits = []
    offset = 0

    while True:
        response = matomo_api_call(
            "Live.getLastVisitsDetails",
            params={
                "period": period,
                "date": date_range,
                "filter_limit": page_size,
                "filter_offset": offset
            }
        )

        if not response:
            break

        if isinstance(response, dict):
            visits = response.get("value", [])
        else:
            visits = response

        if not visits:
            break

        all_visits.extend(visits)

        if len(visits) < page_size:
            break

        offset += page_size

    return all_visits


def parse_dspace_search_url(url):
    """
    Parses a DSpace search URL and extracts textual queries, facet filters and page number.
    """

    if url is None or str(url).strip() == "":
        return None

    parsed_url = urlparse(str(url).strip())

    if "/search" not in parsed_url.path:
        return None

    query_params = parse_qsl(parsed_url.query, keep_blank_values=True)

    query_text = None
    page = None
    filters = []
    other_params = []

    for key, value in query_params:
        value = str(value).strip() if value is not None else ""

        if key == "query":
            if value != "":
                query_text = value

        elif key == "spc.page":
            if value != "":
                page = value

        elif key.startswith("f."):
            filter_name = key.replace("f.", "", 1)
            filter_value = value

            if filter_value.endswith(",equals"):
                filter_value = filter_value[:-len(",equals")]

            if filter_value != "":
                filters.append(f"{filter_name}={filter_value}")

        else:
            if value != "":
                other_params.append(f"{key}={value}")
            else:
                other_params.append(key)

    search_parts = []

    if query_text:
        search_parts.append(f"query={query_text}")

    if filters:
        search_parts.append("filters: " + "; ".join(filters))

    if page:
        search_parts.append(f"page={page}")

    if other_params:
        search_parts.append("other_params: " + "; ".join(other_params))

    if search_parts:
        search_query = " | ".join(search_parts)
    else:
        search_query = "search_without_query_or_filters"

    return {
        "search_query": search_query,
        "search_query_text": query_text,
        "search_filters": "; ".join(filters) if filters else None,
        "search_page": page,
        "search_url": url
    }


def join_unique_non_empty(values):
    """
    Joins unique non-empty string values.
    """

    clean_values = sorted(
        set(
            str(value).strip()
            for value in values
            if value is not None and str(value).strip() != ""
        )
    )

    return "; ".join(clean_values)


visits_reference_month = get_matomo_visit_details(
    date_range=f"{period_start},{period_end}",
    period="range",
    page_size=500
)

visit_reference_month_rows = []

for visit in visits_reference_month:
    action_details = visit.get("actionDetails", [])

    pageviews_count = sum(
        1 for action in action_details
        if action.get("type") == "action"
    )

    detected_searches = []

    for action in action_details:
        action_url = (
            action.get("url")
            or action.get("pageUrl")
            or action.get("urlBase")
        )

        parsed_search = parse_dspace_search_url(action_url)

        if parsed_search is not None:
            detected_searches.append(parsed_search)

    search_queries_detected = join_unique_non_empty(
        [search.get("search_query") for search in detected_searches]
    )

    search_query_texts_detected = join_unique_non_empty(
        [search.get("search_query_text") for search in detected_searches]
    )

    search_filters_detected = join_unique_non_empty(
        [search.get("search_filters") for search in detected_searches]
    )

    search_pages_detected = join_unique_non_empty(
        [search.get("search_page") for search in detected_searches]
    )

    search_urls_detected = join_unique_non_empty(
        [search.get("search_url") for search in detected_searches]
    )

    visit_reference_month_rows.append({
        "visit_id": visit.get("idVisit"),
        "visitor_id": visit.get("visitorId"),
        "visit_first_action_time": visit.get("serverDatePrettyFirstAction"),
        "visit_last_action_time": visit.get("serverDatePretty"),
        "visit_duration_seconds": visit.get("visitDuration"),
        "actions_count": visit.get("actions"),
        "pageviews_count": pageviews_count,
        "searches_count": visit.get("siteSearches"),
        "searches_detected_count": len(detected_searches),
        "search_queries_detected": search_queries_detected,
        "search_query_texts_detected": search_query_texts_detected,
        "search_filters_detected": search_filters_detected,
        "search_pages_detected": search_pages_detected,
        "search_urls_detected": search_urls_detected,
        "downloads_count": visit.get("downloads"),
        "outlinks_count": visit.get("outlinks"),
        "events_count": visit.get("events"),
        "referrer_type": visit.get("referrerType"),
        "referrer_name": visit.get("referrerName"),
        "referrer_url": visit.get("referrerUrl"),
        "country": visit.get("country"),
        "region": visit.get("region"),
        "city": visit.get("city"),
        "latitude": visit.get("latitude"),
        "longitude": visit.get("longitude"),
        "browser": visit.get("browser"),
        "browser_version": visit.get("browserVersion"),
        "operating_system": visit.get("operatingSystem"),
        "device_type": visit.get("deviceType"),
        "device_brand": visit.get("deviceBrand"),
        "device_model": visit.get("deviceModel"),
        "resolution": visit.get("resolution"),
        "language": visit.get("language"),
        "site_id": MATOMO_SITE_ID,
        "period": "range",
        "period_start": period_start,
        "period_end": period_end
    })

df_visits_reference_month = pd.DataFrame(visit_reference_month_rows)

df_visits_reference_month["reference_month"] = REPORT_MONTH
df_visits_reference_month["environment"] = ENVIRONMENT
df_visits_reference_month["source"] = "matomo"
df_visits_reference_month["metric_definition"] = (
    "list of all visits recorded by Matomo for the tracked DSpace site during the "
    "reference month, enriched with DSpace search details parsed from actionDetails URLs. "
    "Referrer fields describe the visit origin, not the internal DSpace search query."
)

output_visits_reference_month = (
    export_path("matomo_visits_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_reference_month, 
    output_visits_reference_month,
    index=False
)

print(
    "Visits in the reference month extracted: "
    f"{len(df_visits_reference_month)}"
)

print(f"Reference period: {period_start} to {period_end}")

if not df_visits_reference_month.empty:
    print("Visits by country:")
    display(
        df_visits_reference_month
        .groupby("country", dropna=False)
        .size()
        .reset_index(name="visits_count")
        .sort_values("visits_count", ascending=False)
        .head(20)
    )

    print("Visits by referrer type:")
    display(
        df_visits_reference_month
        .groupby("referrer_type", dropna=False)
        .size()
        .reset_index(name="visits_count")
        .sort_values("visits_count", ascending=False)
    )

    print("Visits with detected DSpace search actions:")
    display(
        df_visits_reference_month
        .assign(has_search_action=lambda df: df["searches_detected_count"] > 0)
        .groupby("has_search_action", dropna=False)
        .size()
        .reset_index(name="visits_count")
        .sort_values("visits_count", ascending=False)
    )

    print("Most frequent detected search queries:")
    display(
        df_visits_reference_month[
            df_visits_reference_month["search_queries_detected"] != ""
        ]
        .groupby("search_queries_detected", dropna=False)
        .size()
        .reset_index(name="visits_count")
        .sort_values("visits_count", ascending=False)
        .head(20)
    )

display(df_visits_reference_month.head())

print(f"File saved to: {output_visits_reference_month.resolve()}")

### Pageviews in the reference month

In [ ]:
# =========================================================
# PAGEVIEWS IN THE REFERENCE MONTH - ACTION-LEVEL DETAILS
# =========================================================

# - visit_id
# - visitor_id
# - action_position
# - action_type
# - page_title
# - page_url
# - page_path
# - page_query
# - action_server_time
# - action_time_spent_seconds
# - visit_first_action_time
# - visit_last_action_time
# - visit_duration_seconds
# - visit_actions_count
# - referrer_type
# - referrer_name
# - referrer_url
# - country
# - region
# - city
# - browser
# - operating_system
# - device_type
# - language
# - site_id
# - period
# - period_start
# - period_end
# - reference_month
# - environment
# - source
# - metric_definition

from urllib.parse import urlparse

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")


def get_matomo_visit_details(date_range, period="range", page_size=500):
    """
    Retrieves visit-level details from Matomo using Live.getLastVisitsDetails.

    Matomo returns visits in pages using filter_limit and filter_offset.
    """

    all_visits = []
    offset = 0

    while True:
        response = matomo_api_call(
            "Live.getLastVisitsDetails",
            params={
                "period": period,
                "date": date_range,
                "filter_limit": page_size,
                "filter_offset": offset
            }
        )

        if not response:
            break

        if isinstance(response, dict):
            visits = response.get("value", [])
        else:
            visits = response

        if not visits:
            break

        all_visits.extend(visits)

        if len(visits) < page_size:
            break

        offset += page_size

    return all_visits


def parse_url_parts(url):
    """
    Extracts path and query string from a URL.
    """

    if url is None or str(url).strip() == "":
        return {
            "page_path": None,
            "page_query": None
        }

    parsed_url = urlparse(str(url).strip())

    return {
        "page_path": parsed_url.path,
        "page_query": parsed_url.query if parsed_url.query != "" else None
    }


visits_reference_month = get_matomo_visit_details(
    date_range=f"{period_start},{period_end}",
    period="range",
    page_size=500
)

pageview_detail_rows = []

for visit in visits_reference_month:
    action_details = visit.get("actionDetails", [])

    for action_position, action in enumerate(action_details, start=1):
        action_type = action.get("type")

        # In Matomo, normal pageviews usually have type == "action".
        # Other possible action types may include download, outlink, event, etc.
        if action_type != "action":
            continue

        page_url = (
            action.get("url")
            or action.get("pageUrl")
            or action.get("urlBase")
        )

        parsed_url_parts = parse_url_parts(page_url)

        pageview_detail_rows.append({
            "visit_id": visit.get("idVisit"),
            "visitor_id": visit.get("visitorId"),
            "action_position": action_position,
            "action_type": action_type,
            "page_title": action.get("pageTitle") or action.get("title"),
            "page_url": page_url,
            "page_path": parsed_url_parts["page_path"],
            "page_query": parsed_url_parts["page_query"],
            "action_server_time": (
                action.get("serverTimePretty")
                or action.get("serverTimePrettyFirstAction")
                or action.get("timestamp")
            ),
            "action_time_spent_seconds": action.get("timeSpent"),
            "visit_first_action_time": visit.get("serverDatePrettyFirstAction"),
            "visit_last_action_time": visit.get("serverDatePretty"),
            "visit_duration_seconds": visit.get("visitDuration"),
            "visit_actions_count": visit.get("actions"),
            "referrer_type": visit.get("referrerType"),
            "referrer_name": visit.get("referrerName"),
            "referrer_url": visit.get("referrerUrl"),
            "country": visit.get("country"),
            "region": visit.get("region"),
            "city": visit.get("city"),
            "browser": visit.get("browser"),
            "operating_system": visit.get("operatingSystem"),
            "device_type": visit.get("deviceType"),
            "language": visit.get("language"),
            "site_id": MATOMO_SITE_ID,
            "period": "range",
            "period_start": period_start,
            "period_end": period_end
        })

df_pageviews_reference_month_action_details = pd.DataFrame(
    pageview_detail_rows
)

df_pageviews_reference_month_action_details["reference_month"] = REPORT_MONTH
df_pageviews_reference_month_action_details["environment"] = ENVIRONMENT
df_pageviews_reference_month_action_details["source"] = "matomo"
df_pageviews_reference_month_action_details["metric_definition"] = (
    "action-level pageview details recorded by Matomo during the reference month. "
    "Each row corresponds to a single pageview action within a visit, extracted from "
    "Live.getLastVisitsDetails actionDetails."
)

output_pageviews_reference_month_action_details = (
    export_path("matomo_pageviews_reference_month_action_details.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_pageviews_reference_month_action_details, 
    output_pageviews_reference_month_action_details,
    index=False
)

print(
    "Pageview action details in the reference month extracted: "
    f"{len(df_pageviews_reference_month_action_details)}"
)

print(
    "Visits processed: "
    f"{len(visits_reference_month)}"
)

print(f"Reference period: {period_start} to {period_end}")

if not df_pageviews_reference_month_action_details.empty:
    print("Top page paths by raw pageview actions:")
    display(
        df_pageviews_reference_month_action_details
        .groupby("page_path", dropna=False)
        .size()
        .reset_index(name="pageviews_count")
        .sort_values("pageviews_count", ascending=False)
        .head(30)
    )

    print("Top page URLs by raw pageview actions:")
    display(
        df_pageviews_reference_month_action_details
        .groupby("page_url", dropna=False)
        .size()
        .reset_index(name="pageviews_count")
        .sort_values("pageviews_count", ascending=False)
        .head(30)
    )

display(df_pageviews_reference_month_action_details.head(30))

print(f"File saved to: {output_pageviews_reference_month_action_details.resolve()}")

### Average visit duration in the reference month

In [ ]:
# =========================================================
# AVERAGE VISIT DURATION IN THE REFERENCE MONTH
# =========================================================

# - metric_name
# - average_visit_duration_seconds
# - average_visit_duration_minutes
# - period
# - period_start
# - period_end
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")
report_month_date = reference_month_start.strftime("%Y-%m-%d")

response_average_visit_duration_reference_month = matomo_api_call(
    "VisitsSummary.get",
    params={
        "period": "month",
        "date": report_month_date
    }
)

average_visit_duration_seconds_reference_month = (
    response_average_visit_duration_reference_month.get("avg_time_on_site", 0)
)

average_visit_duration_minutes_reference_month = (
    round(average_visit_duration_seconds_reference_month / 60, 2)
    if average_visit_duration_seconds_reference_month is not None
    else 0
)

df_average_visit_duration_reference_month = pd.DataFrame([
    {
        "metric_name": "average_visit_duration_reference_month",
        "average_visit_duration_seconds": average_visit_duration_seconds_reference_month,
        "average_visit_duration_minutes": average_visit_duration_minutes_reference_month,
        "period": "month",
        "period_start": period_start,
        "period_end": period_end,
        "site_id": MATOMO_SITE_ID
    }
])

df_average_visit_duration_reference_month["reference_month"] = REPORT_MONTH
df_average_visit_duration_reference_month["environment"] = ENVIRONMENT
df_average_visit_duration_reference_month["source"] = "matomo"
df_average_visit_duration_reference_month["metric_definition"] = (
    "average visit duration recorded by Matomo for the tracked DSpace site "
    "during the reference month, expressed in seconds and minutes"
)

output_average_visit_duration_reference_month = (
    export_path("matomo_average_visit_duration_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_average_visit_duration_reference_month, 
    output_average_visit_duration_reference_month,
    index=False
)

print(
    "Average visit duration in the reference month: "
    f"{average_visit_duration_seconds_reference_month} seconds "
    f"({average_visit_duration_minutes_reference_month} minutes)"
)

print(f"Reference period: {period_start} to {period_end}")

display(df_average_visit_duration_reference_month)

print(f"File saved to: {output_average_visit_duration_reference_month.resolve()}")

### Bounce rate in the reference month

In [ ]:
# =========================================================
# BOUNCE RATE IN THE REFERENCE MONTH
# =========================================================

# - metric_name
# - bounce_rate
# - bounce_count
# - visits_count
# - period
# - period_start
# - period_end
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")
report_month_date = reference_month_start.strftime("%Y-%m-%d")

response_bounce_rate_reference_month = matomo_api_call(
    "VisitsSummary.get",
    params={
        "period": "month",
        "date": report_month_date
    }
)

bounce_rate_reference_month = (
    response_bounce_rate_reference_month.get("bounce_rate")
)

bounce_count_reference_month = (
    response_bounce_rate_reference_month.get("bounce_count", 0)
)

visits_count_reference_month = (
    response_bounce_rate_reference_month.get("nb_visits", 0)
)

df_bounce_rate_reference_month = pd.DataFrame([
    {
        "metric_name": "bounce_rate_reference_month",
        "bounce_rate": bounce_rate_reference_month,
        "bounce_count": bounce_count_reference_month,
        "visits_count": visits_count_reference_month,
        "period": "month",
        "period_start": period_start,
        "period_end": period_end,
        "site_id": MATOMO_SITE_ID
    }
])

df_bounce_rate_reference_month["reference_month"] = REPORT_MONTH
df_bounce_rate_reference_month["environment"] = ENVIRONMENT
df_bounce_rate_reference_month["source"] = "matomo"
df_bounce_rate_reference_month["metric_definition"] = (
    "percentage of bounced visits recorded by Matomo for the tracked DSpace site "
    "during the reference month. A bounced visit is a visit with only one tracked action."
)

output_bounce_rate_reference_month = (
    export_path("matomo_bounce_rate_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_bounce_rate_reference_month, 
    output_bounce_rate_reference_month,
    index=False
)

print(f"Bounce rate in the reference month: {bounce_rate_reference_month}")
print(f"Bounce count in the reference month: {bounce_count_reference_month}")
print(f"Total visits in the reference month: {visits_count_reference_month}")
print(f"Reference period: {period_start} to {period_end}")

display(df_bounce_rate_reference_month)

print(f"File saved to: {output_bounce_rate_reference_month.resolve()}")

### Actions per visit in the reference month

In [ ]:
# =========================================================
# ACTIONS PER VISIT IN THE REFERENCE MONTH
# =========================================================

# - metric_name
# - actions_per_visit
# - actions_count
# - visits_count
# - period
# - period_start
# - period_end
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")
report_month_date = reference_month_start.strftime("%Y-%m-%d")

response_actions_per_visit_reference_month = matomo_api_call(
    "VisitsSummary.get",
    params={
        "period": "month",
        "date": report_month_date
    }
)

actions_per_visit_reference_month = (
    response_actions_per_visit_reference_month.get("nb_actions_per_visit")
)

actions_count_reference_month = (
    response_actions_per_visit_reference_month.get("nb_actions", 0)
)

visits_count_reference_month = (
    response_actions_per_visit_reference_month.get("nb_visits", 0)
)

df_actions_per_visit_reference_month = pd.DataFrame([
    {
        "metric_name": "actions_per_visit_reference_month",
        "actions_per_visit": actions_per_visit_reference_month,
        "actions_count": actions_count_reference_month,
        "visits_count": visits_count_reference_month,
        "period": "month",
        "period_start": period_start,
        "period_end": period_end,
        "site_id": MATOMO_SITE_ID
    }
])

df_actions_per_visit_reference_month["reference_month"] = REPORT_MONTH
df_actions_per_visit_reference_month["environment"] = ENVIRONMENT
df_actions_per_visit_reference_month["source"] = "matomo"
df_actions_per_visit_reference_month["metric_definition"] = (
    "average number of tracked actions per visit recorded by Matomo for the tracked "
    "DSpace site during the reference month. Actions may include pageviews, downloads, "
    "outlinks, site searches or events, depending on the Matomo tracking configuration."
)

output_actions_per_visit_reference_month = (
    export_path("matomo_actions_per_visit_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_actions_per_visit_reference_month, 
    output_actions_per_visit_reference_month,
    index=False
)

print(
    "Actions per visit in the reference month: "
    f"{actions_per_visit_reference_month}"
)

print(
    "Total actions in the reference month: "
    f"{actions_count_reference_month}"
)

print(
    "Total visits in the reference month: "
    f"{visits_count_reference_month}"
)

print(f"Reference period: {period_start} to {period_end}")

display(df_actions_per_visit_reference_month)

print(f"File saved to: {output_actions_per_visit_reference_month.resolve()}")

## **Geography**

### Visits by country

In [ ]:
# =========================================================
# VISITS BY COUNTRY
# =========================================================

# - country
# - country_code
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_visits_by_country = matomo_api_call(
    "UserCountry.getCountry",
    params={
        "period": "range",
        "date": "2000-01-01,today",
        "filter_limit": -1
    }
)

visits_by_country_rows = []

for row in response_visits_by_country:
    visits_by_country_rows.append({
        "country": row.get("label"),
        "country_code": row.get("code"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "range",
        "site_id": MATOMO_SITE_ID
    })

df_visits_by_country = pd.DataFrame(visits_by_country_rows)

if not df_visits_by_country.empty:
    df_visits_by_country = (
        df_visits_by_country
        .sort_values("visits_count", ascending=False)
    )

df_visits_by_country["reference_month"] = REPORT_MONTH
df_visits_by_country["environment"] = ENVIRONMENT
df_visits_by_country["source"] = "matomo"
df_visits_by_country["metric_definition"] = (
    "visits grouped by visitor country recorded by Matomo for the tracked DSpace site "
    "over the full available reporting range"
)

output_visits_by_country = export_path("matomo_visits_by_country.csv")

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_by_country, output_visits_by_country, index=False)

print(f"Countries with visits: {len(df_visits_by_country)}")
print(
    "Total visits by country: "
    f"{df_visits_by_country['visits_count'].sum() if not df_visits_by_country.empty else 0}"
)

display(df_visits_by_country.head(30))

print(f"File saved to: {output_visits_by_country.resolve()}")

### Visits by country in the reference month

In [ ]:
# =========================================================
# VISITS BY COUNTRY IN THE REFERENCE MONTH
# =========================================================

# - country
# - country_code
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - period_start
# - period_end
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")
report_month_date = reference_month_start.strftime("%Y-%m-%d")

response_visits_by_country_reference_month = matomo_api_call(
    "UserCountry.getCountry",
    params={
        "period": "month",
        "date": report_month_date,
        "filter_limit": -1
    }
)

visits_by_country_reference_month_rows = []

for row in response_visits_by_country_reference_month:
    visits_by_country_reference_month_rows.append({
        "country": row.get("label"),
        "country_code": row.get("code"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "month",
        "period_start": period_start,
        "period_end": period_end,
        "site_id": MATOMO_SITE_ID
    })

df_visits_by_country_reference_month = pd.DataFrame(
    visits_by_country_reference_month_rows
)

if not df_visits_by_country_reference_month.empty:
    df_visits_by_country_reference_month = (
        df_visits_by_country_reference_month
        .sort_values("visits_count", ascending=False)
    )

df_visits_by_country_reference_month["reference_month"] = REPORT_MONTH
df_visits_by_country_reference_month["environment"] = ENVIRONMENT
df_visits_by_country_reference_month["source"] = "matomo"
df_visits_by_country_reference_month["metric_definition"] = (
    "visits grouped by visitor country recorded by Matomo for the tracked DSpace site "
    "during the reference month"
)

output_visits_by_country_reference_month = (
    export_path("matomo_visits_by_country_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_by_country_reference_month, 
    output_visits_by_country_reference_month,
    index=False
)

print(
    "Countries with visits in the reference month: "
    f"{len(df_visits_by_country_reference_month)}"
)

print(
    "Total visits by country in the reference month: "
    f"{df_visits_by_country_reference_month['visits_count'].sum() if not df_visits_by_country_reference_month.empty else 0}"
)

print(f"Reference period: {period_start} to {period_end}")

display(df_visits_by_country_reference_month.head(30))

print(f"File saved to: {output_visits_by_country_reference_month.resolve()}")

### Visits by city

In [ ]:
# =========================================================
# VISITS BY CITY
# =========================================================

# - country
# - country_code
# - region
# - city
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_visits_by_city = matomo_api_call(
    "UserCountry.getCity",
    params={
        "period": "range",
        "date": "2000-01-01,today",
        "filter_limit": -1
    }
)

visits_by_city_rows = []

for row in response_visits_by_city:
    visits_by_city_rows.append({
        "country": row.get("country"),
        "country_code": row.get("countryCode"),
        "region": row.get("region"),
        "city": row.get("label"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "range",
        "site_id": MATOMO_SITE_ID
    })

df_visits_by_city = pd.DataFrame(visits_by_city_rows)

if not df_visits_by_city.empty:
    df_visits_by_city = (
        df_visits_by_city
        .sort_values("visits_count", ascending=False)
    )

df_visits_by_city["reference_month"] = REPORT_MONTH
df_visits_by_city["environment"] = ENVIRONMENT
df_visits_by_city["source"] = "matomo"
df_visits_by_city["metric_definition"] = (
    "visits grouped by visitor city recorded by Matomo for the tracked DSpace site "
    "over the full available reporting range"
)

output_visits_by_city = export_path("matomo_visits_by_city.csv")

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_by_city, output_visits_by_city, index=False)

print(f"Cities with visits: {len(df_visits_by_city)}")
print(
    "Total visits by city: "
    f"{df_visits_by_city['visits_count'].sum() if not df_visits_by_city.empty else 0}"
)

display(df_visits_by_city.head(30))

print(f"File saved to: {output_visits_by_city.resolve()}")

### Visits by city in the reference month

In [ ]:
# =========================================================
# VISITS BY CITY IN THE REFERENCE MONTH
# =========================================================

# - country
# - country_code
# - region
# - city
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - period_start
# - period_end
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")
report_month_date = reference_month_start.strftime("%Y-%m-%d")

response_visits_by_city_reference_month = matomo_api_call(
    "UserCountry.getCity",
    params={
        "period": "month",
        "date": report_month_date,
        "filter_limit": -1
    }
)

visits_by_city_reference_month_rows = []

for row in response_visits_by_city_reference_month:
    visits_by_city_reference_month_rows.append({
        "country": row.get("country"),
        "country_code": row.get("countryCode"),
        "region": row.get("region"),
        "city": row.get("label"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "month",
        "period_start": period_start,
        "period_end": period_end,
        "site_id": MATOMO_SITE_ID
    })

df_visits_by_city_reference_month = pd.DataFrame(
    visits_by_city_reference_month_rows
)

if not df_visits_by_city_reference_month.empty:
    df_visits_by_city_reference_month = (
        df_visits_by_city_reference_month
        .sort_values("visits_count", ascending=False)
    )

df_visits_by_city_reference_month["reference_month"] = REPORT_MONTH
df_visits_by_city_reference_month["environment"] = ENVIRONMENT
df_visits_by_city_reference_month["source"] = "matomo"
df_visits_by_city_reference_month["metric_definition"] = (
    "visits grouped by visitor city recorded by Matomo for the tracked DSpace site "
    "during the reference month"
)

output_visits_by_city_reference_month = (
    export_path("matomo_visits_by_city_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_by_city_reference_month, 
    output_visits_by_city_reference_month,
    index=False
)

print(
    "Cities with visits in the reference month: "
    f"{len(df_visits_by_city_reference_month)}"
)

print(
    "Total visits by city in the reference month: "
    f"{df_visits_by_city_reference_month['visits_count'].sum() if not df_visits_by_city_reference_month.empty else 0}"
)

print(f"Reference period: {period_start} to {period_end}")

display(df_visits_by_city_reference_month.head(30))

print(f"File saved to: {output_visits_by_city_reference_month.resolve()}")

### Visits by continent

In [ ]:
# =========================================================
# VISITS BY CONTINENT
# =========================================================

# - continent
# - continent_code
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_visits_by_continent = matomo_api_call(
    "UserCountry.getContinent",
    params={
        "period": "range",
        "date": "2000-01-01,today",
        "filter_limit": -1
    }
)

visits_by_continent_rows = []

for row in response_visits_by_continent:
    visits_by_continent_rows.append({
        "continent": row.get("label"),
        "continent_code": row.get("code"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "range",
        "site_id": MATOMO_SITE_ID
    })

df_visits_by_continent = pd.DataFrame(visits_by_continent_rows)

if not df_visits_by_continent.empty:
    df_visits_by_continent = (
        df_visits_by_continent
        .sort_values("visits_count", ascending=False)
    )

df_visits_by_continent["reference_month"] = REPORT_MONTH
df_visits_by_continent["environment"] = ENVIRONMENT
df_visits_by_continent["source"] = "matomo"
df_visits_by_continent["metric_definition"] = (
    "visits grouped by visitor continent recorded by Matomo for the tracked DSpace site "
    "over the full available reporting range"
)

output_visits_by_continent = export_path("matomo_visits_by_continent.csv")

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_by_continent, output_visits_by_continent, index=False)

print(f"Continents with visits: {len(df_visits_by_continent)}")
print(
    "Total visits by continent: "
    f"{df_visits_by_continent['visits_count'].sum() if not df_visits_by_continent.empty else 0}"
)

display(df_visits_by_continent)

print(f"File saved to: {output_visits_by_continent.resolve()}")

### Visits by continent in the reference month

In [ ]:
# =========================================================
# VISITS BY CONTINENT IN THE REFERENCE MONTH
# =========================================================

# - continent
# - continent_code
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - period_start
# - period_end
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")
report_month_date = reference_month_start.strftime("%Y-%m-%d")

response_visits_by_continent_reference_month = matomo_api_call(
    "UserCountry.getContinent",
    params={
        "period": "month",
        "date": report_month_date,
        "filter_limit": -1
    }
)

visits_by_continent_reference_month_rows = []

for row in response_visits_by_continent_reference_month:
    visits_by_continent_reference_month_rows.append({
        "continent": row.get("label"),
        "continent_code": row.get("code"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "month",
        "period_start": period_start,
        "period_end": period_end,
        "site_id": MATOMO_SITE_ID
    })

df_visits_by_continent_reference_month = pd.DataFrame(
    visits_by_continent_reference_month_rows
)

if not df_visits_by_continent_reference_month.empty:
    df_visits_by_continent_reference_month = (
        df_visits_by_continent_reference_month
        .sort_values("visits_count", ascending=False)
    )

df_visits_by_continent_reference_month["reference_month"] = REPORT_MONTH
df_visits_by_continent_reference_month["environment"] = ENVIRONMENT
df_visits_by_continent_reference_month["source"] = "matomo"
df_visits_by_continent_reference_month["metric_definition"] = (
    "visits grouped by visitor continent recorded by Matomo for the tracked DSpace site "
    "during the reference month"
)

output_visits_by_continent_reference_month = (
    export_path("matomo_visits_by_continent_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_by_continent_reference_month, 
    output_visits_by_continent_reference_month,
    index=False
)

print(
    "Continents with visits in the reference month: "
    f"{len(df_visits_by_continent_reference_month)}"
)

print(
    "Total visits by continent in the reference month: "
    f"{df_visits_by_continent_reference_month['visits_count'].sum() if not df_visits_by_continent_reference_month.empty else 0}"
)

print(f"Reference period: {period_start} to {period_end}")

display(df_visits_by_continent_reference_month)

print(f"File saved to: {output_visits_by_continent_reference_month.resolve()}")

## **Referrers and Acquisition**

### Direct visits

In [ ]:
# =========================================================
# DIRECT VISITS
# =========================================================

# - metric_name
# - referrer_type
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_referrer_types = matomo_api_call(
    "Referrers.getReferrerType",
    params={
        "period": "range",
        "date": "2000-01-01,today",
        "filter_limit": -1
    }
)

direct_visits_rows = []

for row in response_referrer_types:
    referrer_type_label = str(row.get("label", "")).lower()

    if "direct" in referrer_type_label:
        direct_visits_rows.append({
            "metric_name": "direct_visits",
            "referrer_type": row.get("label"),
            "visits_count": row.get("nb_visits", 0),
            "unique_visitors_count": row.get("nb_uniq_visitors", 0),
            "actions_count": row.get("nb_actions", 0),
            "bounce_count": row.get("bounce_count", 0),
            "bounce_rate": row.get("bounce_rate"),
            "actions_per_visit": row.get("nb_actions_per_visit"),
            "average_time_on_site": row.get("avg_time_on_site"),
            "period": "range",
            "site_id": MATOMO_SITE_ID
        })

df_direct_visits = pd.DataFrame(direct_visits_rows)

df_direct_visits["reference_month"] = REPORT_MONTH
df_direct_visits["environment"] = ENVIRONMENT
df_direct_visits["source"] = "matomo"
df_direct_visits["metric_definition"] = (
    "visits whose referrer type is classified by Matomo as direct entry. "
    "Direct visits usually correspond to users typing the URL, using bookmarks, "
    "opening links from applications that do not pass a referrer, or visits where "
    "the referrer information is unavailable."
)

output_direct_visits = export_path("matomo_direct_visits.csv")

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_direct_visits, output_direct_visits, index=False)

direct_visits_count = (
    df_direct_visits["visits_count"].sum()
    if not df_direct_visits.empty
    else 0
)

print(f"Direct visits: {direct_visits_count}")

display(df_direct_visits)

print(f"File saved to: {output_direct_visits.resolve()}")

### Direct visits in the reference month

In [ ]:
# =========================================================
# DIRECT VISITS IN THE REFERENCE MONTH - WITH DETAILS
# =========================================================

# - visit_id
# - visitor_id
# - visit_first_action_time
# - visit_last_action_time
# - visit_duration_seconds
# - actions_count
# - pageviews_count
# - searches_count
# - downloads_count
# - outlinks_count
# - events_count
# - referrer_type
# - referrer_name
# - referrer_url
# - country
# - region
# - city
# - latitude
# - longitude
# - browser
# - browser_version
# - operating_system
# - device_type
# - device_brand
# - device_model
# - resolution
# - language
# - site_id
# - period
# - period_start
# - period_end
# - reference_month
# - environment
# - source
# - metric_definition

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")


def get_matomo_visit_details(date_range, period="range", page_size=500):
    """
    Retrieves visit-level details from Matomo using Live.getLastVisitsDetails.

    Matomo returns visits in pages using filter_limit and filter_offset.
    """

    all_visits = []
    offset = 0

    while True:
        response = matomo_api_call(
            "Live.getLastVisitsDetails",
            params={
                "period": period,
                "date": date_range,
                "filter_limit": page_size,
                "filter_offset": offset
            }
        )

        if not response:
            break

        if isinstance(response, dict):
            visits = response.get("value", [])
        else:
            visits = response

        if not visits:
            break

        all_visits.extend(visits)

        if len(visits) < page_size:
            break

        offset += page_size

    return all_visits


def is_direct_visit(visit):
    """
    Checks whether a Matomo visit is classified as direct.

    Matomo usually represents direct entries with referrerType values such as:
    - direct
    - direct entry
    Depending on the API response, the label may vary slightly.
    """

    referrer_type = str(visit.get("referrerType", "")).lower().strip()
    referrer_name = str(visit.get("referrerName", "")).lower().strip()
    referrer_url = str(visit.get("referrerUrl", "")).lower().strip()

    if "direct" in referrer_type:
        return True

    if "direct" in referrer_name:
        return True

    if referrer_type == "" and referrer_url == "":
        return True

    return False


visits_reference_month = get_matomo_visit_details(
    date_range=f"{period_start},{period_end}",
    period="range",
    page_size=500
)

direct_visit_reference_month_rows = []

for visit in visits_reference_month:
    if not is_direct_visit(visit):
        continue

    action_details = visit.get("actionDetails", [])

    pageviews_count = sum(
        1 for action in action_details
        if action.get("type") == "action"
    )

    direct_visit_reference_month_rows.append({
        "visit_id": visit.get("idVisit"),
        "visitor_id": visit.get("visitorId"),
        "visit_first_action_time": visit.get("serverDatePrettyFirstAction"),
        "visit_last_action_time": visit.get("serverDatePretty"),
        "visit_duration_seconds": visit.get("visitDuration"),
        "actions_count": visit.get("actions"),
        "pageviews_count": pageviews_count,
        "searches_count": visit.get("siteSearches"),
        "downloads_count": visit.get("downloads"),
        "outlinks_count": visit.get("outlinks"),
        "events_count": visit.get("events"),
        "referrer_type": visit.get("referrerType"),
        "referrer_name": visit.get("referrerName"),
        "referrer_url": visit.get("referrerUrl"),
        "country": visit.get("country"),
        "region": visit.get("region"),
        "city": visit.get("city"),
        "latitude": visit.get("latitude"),
        "longitude": visit.get("longitude"),
        "browser": visit.get("browser"),
        "browser_version": visit.get("browserVersion"),
        "operating_system": visit.get("operatingSystem"),
        "device_type": visit.get("deviceType"),
        "device_brand": visit.get("deviceBrand"),
        "device_model": visit.get("deviceModel"),
        "resolution": visit.get("resolution"),
        "language": visit.get("language"),
        "site_id": MATOMO_SITE_ID,
        "period": "range",
        "period_start": period_start,
        "period_end": period_end
    })

df_direct_visits_reference_month_details = pd.DataFrame(
    direct_visit_reference_month_rows
)

df_direct_visits_reference_month_details["reference_month"] = REPORT_MONTH
df_direct_visits_reference_month_details["environment"] = ENVIRONMENT
df_direct_visits_reference_month_details["source"] = "matomo"
df_direct_visits_reference_month_details["metric_definition"] = (
    "visit-level details for visits classified as direct entries by Matomo during "
    "the reference month. Direct visits usually correspond to users typing the URL, "
    "using bookmarks, opening links from applications that do not pass a referrer, "
    "or visits where referrer information is unavailable."
)

output_direct_visits_reference_month_details = (
    export_path("matomo_direct_visits_reference_month_details.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_direct_visits_reference_month_details, 
    output_direct_visits_reference_month_details,
    index=False
)

print(
    "Direct visits in the reference month extracted: "
    f"{len(df_direct_visits_reference_month_details)}"
)

print(
    "All visits processed in the reference month: "
    f"{len(visits_reference_month)}"
)

print(f"Reference period: {period_start} to {period_end}")

if not df_direct_visits_reference_month_details.empty:
    print("Direct visits by country:")
    display(
        df_direct_visits_reference_month_details
        .groupby("country", dropna=False)
        .size()
        .reset_index(name="direct_visits_count")
        .sort_values("direct_visits_count", ascending=False)
        .head(20)
    )

    print("Direct visits by city:")
    display(
        df_direct_visits_reference_month_details
        .groupby(["country", "city"], dropna=False)
        .size()
        .reset_index(name="direct_visits_count")
        .sort_values("direct_visits_count", ascending=False)
        .head(20)
    )

    print("Direct visits by browser:")
    display(
        df_direct_visits_reference_month_details
        .groupby("browser", dropna=False)
        .size()
        .reset_index(name="direct_visits_count")
        .sort_values("direct_visits_count", ascending=False)
        .head(20)
    )

    print("Direct visits by device type:")
    display(
        df_direct_visits_reference_month_details
        .groupby("device_type", dropna=False)
        .size()
        .reset_index(name="direct_visits_count")
        .sort_values("direct_visits_count", ascending=False)
    )

display(df_direct_visits_reference_month_details.head(30))

print(f"File saved to: {output_direct_visits_reference_month_details.resolve()}")

### Visits from search engines

In [ ]:
# =========================================================
# VISITS FROM SEARCH ENGINES
# =========================================================

# - search_engine
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_visits_from_search_engines = matomo_api_call(
    "Referrers.getSearchEngines",
    params={
        "period": "range",
        "date": "2000-01-01,today",
        "filter_limit": -1
    }
)

visits_from_search_engines_rows = []

for row in response_visits_from_search_engines:
    visits_from_search_engines_rows.append({
        "search_engine": row.get("label"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "range",
        "site_id": MATOMO_SITE_ID
    })

df_visits_from_search_engines = pd.DataFrame(
    visits_from_search_engines_rows
)

if not df_visits_from_search_engines.empty:
    df_visits_from_search_engines = (
        df_visits_from_search_engines
        .sort_values("visits_count", ascending=False)
    )

df_visits_from_search_engines["reference_month"] = REPORT_MONTH
df_visits_from_search_engines["environment"] = ENVIRONMENT
df_visits_from_search_engines["source"] = "matomo"
df_visits_from_search_engines["metric_definition"] = (
    "visits referred by search engines and recorded by Matomo for the tracked "
    "DSpace site over the full available reporting range"
)

output_visits_from_search_engines = (
    export_path("matomo_visits_from_search_engines.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_from_search_engines, 
    output_visits_from_search_engines,
    index=False
)

print(
    "Search engines with visits: "
    f"{len(df_visits_from_search_engines)}"
)

print(
    "Total visits from search engines: "
    f"{df_visits_from_search_engines['visits_count'].sum() if not df_visits_from_search_engines.empty else 0}"
)

display(df_visits_from_search_engines.head(30))

print(f"File saved to: {output_visits_from_search_engines.resolve()}")

### Visits from search engines in the reference month

In [ ]:
# =========================================================
# VISITS FROM SEARCH ENGINES IN THE REFERENCE MONTH
# =========================================================

# - search_engine
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - period_start
# - period_end
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")
report_month_date = reference_month_start.strftime("%Y-%m-%d")

response_visits_from_search_engines_reference_month = matomo_api_call(
    "Referrers.getSearchEngines",
    params={
        "period": "month",
        "date": report_month_date,
        "filter_limit": -1
    }
)

visits_from_search_engines_reference_month_rows = []

for row in response_visits_from_search_engines_reference_month:
    visits_from_search_engines_reference_month_rows.append({
        "search_engine": row.get("label"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "month",
        "period_start": period_start,
        "period_end": period_end,
        "site_id": MATOMO_SITE_ID
    })

df_visits_from_search_engines_reference_month = pd.DataFrame(
    visits_from_search_engines_reference_month_rows
)

if not df_visits_from_search_engines_reference_month.empty:
    df_visits_from_search_engines_reference_month = (
        df_visits_from_search_engines_reference_month
        .sort_values("visits_count", ascending=False)
    )

df_visits_from_search_engines_reference_month["reference_month"] = REPORT_MONTH
df_visits_from_search_engines_reference_month["environment"] = ENVIRONMENT
df_visits_from_search_engines_reference_month["source"] = "matomo"
df_visits_from_search_engines_reference_month["metric_definition"] = (
    "visits referred by search engines and recorded by Matomo for the tracked "
    "DSpace site during the reference month"
)

output_visits_from_search_engines_reference_month = (
    export_path("matomo_visits_from_search_engines_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_from_search_engines_reference_month, 
    output_visits_from_search_engines_reference_month,
    index=False
)

print(
    "Search engines with visits in the reference month: "
    f"{len(df_visits_from_search_engines_reference_month)}"
)

print(
    "Total visits from search engines in the reference month: "
    f"{df_visits_from_search_engines_reference_month['visits_count'].sum() if not df_visits_from_search_engines_reference_month.empty else 0}"
)

print(f"Reference period: {period_start} to {period_end}")

display(df_visits_from_search_engines_reference_month.head(30))

print(f"File saved to: {output_visits_from_search_engines_reference_month.resolve()}")

### Visits from websites

In [ ]:
# =========================================================
# VISITS FROM WEBSITES
# =========================================================

# - website
# - website_url
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_visits_from_websites = matomo_api_call(
    "Referrers.getWebsites",
    params={
        "period": "range",
        "date": "2000-01-01,today",
        "filter_limit": -1
    }
)

visits_from_websites_rows = []

for row in response_visits_from_websites:
    visits_from_websites_rows.append({
        "website": row.get("label"),
        "website_url": row.get("url"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "range",
        "site_id": MATOMO_SITE_ID
    })

df_visits_from_websites = pd.DataFrame(visits_from_websites_rows)

if not df_visits_from_websites.empty:
    df_visits_from_websites = (
        df_visits_from_websites
        .sort_values("visits_count", ascending=False)
    )

df_visits_from_websites["reference_month"] = REPORT_MONTH
df_visits_from_websites["environment"] = ENVIRONMENT
df_visits_from_websites["source"] = "matomo"
df_visits_from_websites["metric_definition"] = (
    "visits referred by external websites and recorded by Matomo for the tracked "
    "DSpace site over the full available reporting range"
)

output_visits_from_websites = (
    export_path("matomo_visits_from_websites.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_from_websites, 
    output_visits_from_websites,
    index=False
)

print(
    "Websites with referred visits: "
    f"{len(df_visits_from_websites)}"
)

print(
    "Total visits from websites: "
    f"{df_visits_from_websites['visits_count'].sum() if not df_visits_from_websites.empty else 0}"
)

display(df_visits_from_websites.head(30))

print(f"File saved to: {output_visits_from_websites.resolve()}")

### Visits from websites in the reference month

In [ ]:
# =========================================================
# VISITS FROM WEBSITES IN THE REFERENCE MONTH
# =========================================================

# - website
# - website_url
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - period_start
# - period_end
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")
report_month_date = reference_month_start.strftime("%Y-%m-%d")

response_visits_from_websites_reference_month = matomo_api_call(
    "Referrers.getWebsites",
    params={
        "period": "month",
        "date": report_month_date,
        "filter_limit": -1
    }
)

visits_from_websites_reference_month_rows = []

for row in response_visits_from_websites_reference_month:
    visits_from_websites_reference_month_rows.append({
        "website": row.get("label"),
        "website_url": row.get("url"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "month",
        "period_start": period_start,
        "period_end": period_end,
        "site_id": MATOMO_SITE_ID
    })

df_visits_from_websites_reference_month = pd.DataFrame(
    visits_from_websites_reference_month_rows
)

if not df_visits_from_websites_reference_month.empty:
    df_visits_from_websites_reference_month = (
        df_visits_from_websites_reference_month
        .sort_values("visits_count", ascending=False)
    )

df_visits_from_websites_reference_month["reference_month"] = REPORT_MONTH
df_visits_from_websites_reference_month["environment"] = ENVIRONMENT
df_visits_from_websites_reference_month["source"] = "matomo"
df_visits_from_websites_reference_month["metric_definition"] = (
    "visits referred by external websites and recorded by Matomo for the tracked "
    "DSpace site during the reference month"
)

output_visits_from_websites_reference_month = (
    export_path("matomo_visits_from_websites_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_from_websites_reference_month, 
    output_visits_from_websites_reference_month,
    index=False
)

print(
    "Websites with referred visits in the reference month: "
    f"{len(df_visits_from_websites_reference_month)}"
)

print(
    "Total visits from websites in the reference month: "
    f"{df_visits_from_websites_reference_month['visits_count'].sum() if not df_visits_from_websites_reference_month.empty else 0}"
)

print(f"Reference period: {period_start} to {period_end}")

display(df_visits_from_websites_reference_month.head(30))

print(f"File saved to: {output_visits_from_websites_reference_month.resolve()}")

### Visits from social networks

In [ ]:
# =========================================================
# VISITS FROM SOCIAL NETWORKS
# =========================================================

# - social_network
# - social_network_url
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_visits_from_social_networks = matomo_api_call(
    "Referrers.getSocials",
    params={
        "period": "range",
        "date": "2000-01-01,today",
        "filter_limit": -1
    }
)

visits_from_social_networks_rows = []

for row in response_visits_from_social_networks:
    visits_from_social_networks_rows.append({
        "social_network": row.get("label"),
        "social_network_url": row.get("url"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "range",
        "site_id": MATOMO_SITE_ID
    })

df_visits_from_social_networks = pd.DataFrame(
    visits_from_social_networks_rows
)

if not df_visits_from_social_networks.empty:
    df_visits_from_social_networks = (
        df_visits_from_social_networks
        .sort_values("visits_count", ascending=False)
    )

df_visits_from_social_networks["reference_month"] = REPORT_MONTH
df_visits_from_social_networks["environment"] = ENVIRONMENT
df_visits_from_social_networks["source"] = "matomo"
df_visits_from_social_networks["metric_definition"] = (
    "visits referred by social networks and recorded by Matomo for the tracked "
    "DSpace site over the full available reporting range"
)

output_visits_from_social_networks = (
    export_path("matomo_visits_from_social_networks.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_from_social_networks, 
    output_visits_from_social_networks,
    index=False
)

print(
    "Social networks with referred visits: "
    f"{len(df_visits_from_social_networks)}"
)

print(
    "Total visits from social networks: "
    f"{df_visits_from_social_networks['visits_count'].sum() if not df_visits_from_social_networks.empty else 0}"
)

display(df_visits_from_social_networks.head(30))

print(f"File saved to: {output_visits_from_social_networks.resolve()}")

### Visits from social networks in the reference month

In [ ]:
# =========================================================
# VISITS FROM SOCIAL NETWORKS IN THE REFERENCE MONTH
# =========================================================

# - social_network
# - social_network_url
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - period_start
# - period_end
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")
report_month_date = reference_month_start.strftime("%Y-%m-%d")

response_visits_from_social_networks_reference_month = matomo_api_call(
    "Referrers.getSocials",
    params={
        "period": "month",
        "date": report_month_date,
        "filter_limit": -1
    }
)

visits_from_social_networks_reference_month_rows = []

for row in response_visits_from_social_networks_reference_month:
    visits_from_social_networks_reference_month_rows.append({
        "social_network": row.get("label"),
        "social_network_url": row.get("url"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "month",
        "period_start": period_start,
        "period_end": period_end,
        "site_id": MATOMO_SITE_ID
    })

df_visits_from_social_networks_reference_month = pd.DataFrame(
    visits_from_social_networks_reference_month_rows
)

if not df_visits_from_social_networks_reference_month.empty:
    df_visits_from_social_networks_reference_month = (
        df_visits_from_social_networks_reference_month
        .sort_values("visits_count", ascending=False)
    )

df_visits_from_social_networks_reference_month["reference_month"] = REPORT_MONTH
df_visits_from_social_networks_reference_month["environment"] = ENVIRONMENT
df_visits_from_social_networks_reference_month["source"] = "matomo"
df_visits_from_social_networks_reference_month["metric_definition"] = (
    "visits referred by social networks and recorded by Matomo for the tracked "
    "DSpace site during the reference month"
)

output_visits_from_social_networks_reference_month = (
    export_path("matomo_visits_from_social_networks_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_from_social_networks_reference_month, 
    output_visits_from_social_networks_reference_month,
    index=False
)

print(
    "Social networks with referred visits in the reference month: "
    f"{len(df_visits_from_social_networks_reference_month)}"
)

print(
    "Total visits from social networks in the reference month: "
    f"{df_visits_from_social_networks_reference_month['visits_count'].sum() if not df_visits_from_social_networks_reference_month.empty else 0}"
)

print(f"Reference period: {period_start} to {period_end}")

display(df_visits_from_social_networks_reference_month.head(30))

print(f"File saved to: {output_visits_from_social_networks_reference_month.resolve()}")

### Campaign visits

In [ ]:
# =========================================================
# CAMPAIGN VISITS
# =========================================================

# - campaign
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_campaign_visits = matomo_api_call(
    "Referrers.getCampaigns",
    params={
        "period": "range",
        "date": "2000-01-01,today",
        "filter_limit": -1
    }
)

campaign_visits_rows = []

for row in response_campaign_visits:
    campaign_visits_rows.append({
        "campaign": row.get("label"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "range",
        "site_id": MATOMO_SITE_ID
    })

df_campaign_visits = pd.DataFrame(campaign_visits_rows)

if not df_campaign_visits.empty:
    df_campaign_visits = (
        df_campaign_visits
        .sort_values("visits_count", ascending=False)
    )

df_campaign_visits["reference_month"] = REPORT_MONTH
df_campaign_visits["environment"] = ENVIRONMENT
df_campaign_visits["source"] = "matomo"
df_campaign_visits["metric_definition"] = (
    "visits attributed by Matomo to campaign parameters over the full available "
    "reporting range. Campaign visits are available only when incoming URLs include "
    "campaign tracking parameters recognized by Matomo."
)

output_campaign_visits = export_path("matomo_campaign_visits.csv")

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_campaign_visits, output_campaign_visits, index=False)

print(f"Campaigns with visits: {len(df_campaign_visits)}")
print(
    "Total campaign visits: "
    f"{df_campaign_visits['visits_count'].sum() if not df_campaign_visits.empty else 0}"
)

display(df_campaign_visits.head(30))

print(f"File saved to: {output_campaign_visits.resolve()}")

### Campaign visits in the reference month

In [ ]:
# =========================================================
# CAMPAIGN VISITS IN THE REFERENCE MONTH
# =========================================================

# - campaign
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - period_start
# - period_end
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")
report_month_date = reference_month_start.strftime("%Y-%m-%d")

response_campaign_visits_reference_month = matomo_api_call(
    "Referrers.getCampaigns",
    params={
        "period": "month",
        "date": report_month_date,
        "filter_limit": -1
    }
)

campaign_visits_reference_month_rows = []

for row in response_campaign_visits_reference_month:
    campaign_visits_reference_month_rows.append({
        "campaign": row.get("label"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "month",
        "period_start": period_start,
        "period_end": period_end,
        "site_id": MATOMO_SITE_ID
    })

df_campaign_visits_reference_month = pd.DataFrame(
    campaign_visits_reference_month_rows
)

if not df_campaign_visits_reference_month.empty:
    df_campaign_visits_reference_month = (
        df_campaign_visits_reference_month
        .sort_values("visits_count", ascending=False)
    )

df_campaign_visits_reference_month["reference_month"] = REPORT_MONTH
df_campaign_visits_reference_month["environment"] = ENVIRONMENT
df_campaign_visits_reference_month["source"] = "matomo"
df_campaign_visits_reference_month["metric_definition"] = (
    "visits attributed by Matomo to campaign parameters during the reference month. "
    "Campaign visits are available only when incoming URLs include campaign tracking "
    "parameters recognized by Matomo."
)

output_campaign_visits_reference_month = (
    export_path("matomo_campaign_visits_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_campaign_visits_reference_month, 
    output_campaign_visits_reference_month,
    index=False
)

print(
    "Campaigns with visits in the reference month: "
    f"{len(df_campaign_visits_reference_month)}"
)

print(
    "Total campaign visits in the reference month: "
    f"{df_campaign_visits_reference_month['visits_count'].sum() if not df_campaign_visits_reference_month.empty else 0}"
)

print(f"Reference period: {period_start} to {period_end}")

display(df_campaign_visits_reference_month.head(30))

print(f"File saved to: {output_campaign_visits_reference_month.resolve()}")

### Top referrer websites

In [ ]:
# =========================================================
# TOP REFERRER WEBSITES
# =========================================================

# - referrer_website
# - referrer_url
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_top_referrer_websites = matomo_api_call(
    "Referrers.getWebsites",
    params={
        "period": "range",
        "date": "2000-01-01,today",
        "filter_limit": -1
    }
)

top_referrer_websites_rows = []

for row in response_top_referrer_websites:
    top_referrer_websites_rows.append({
        "referrer_website": row.get("label"),
        "referrer_url": row.get("url"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "range",
        "site_id": MATOMO_SITE_ID
    })

df_top_referrer_websites = pd.DataFrame(top_referrer_websites_rows)

if not df_top_referrer_websites.empty:
    df_top_referrer_websites = (
        df_top_referrer_websites
        .sort_values("visits_count", ascending=False)
    )

df_top_referrer_websites["reference_month"] = REPORT_MONTH
df_top_referrer_websites["environment"] = ENVIRONMENT
df_top_referrer_websites["source"] = "matomo"
df_top_referrer_websites["metric_definition"] = (
    "top external websites that referred visits to the tracked DSpace site, "
    "as classified by Matomo over the full available reporting range"
)

output_top_referrer_websites = (
    export_path("matomo_top_referrer_websites.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_top_referrer_websites, 
    output_top_referrer_websites,
    index=False
)

print(
    "Top referrer websites extracted: "
    f"{len(df_top_referrer_websites)}"
)

print(
    "Total visits from referrer websites: "
    f"{df_top_referrer_websites['visits_count'].sum() if not df_top_referrer_websites.empty else 0}"
)

display(df_top_referrer_websites.head(30))

print(f"File saved to: {output_top_referrer_websites.resolve()}")

### Top referrer websites in the reference month

In [ ]:
# =========================================================
# TOP REFERRER WEBSITES IN THE REFERENCE MONTH
# =========================================================

# - referrer_website
# - referrer_url
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - period_start
# - period_end
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")
report_month_date = reference_month_start.strftime("%Y-%m-%d")

response_top_referrer_websites_reference_month = matomo_api_call(
    "Referrers.getWebsites",
    params={
        "period": "month",
        "date": report_month_date,
        "filter_limit": -1
    }
)

top_referrer_websites_reference_month_rows = []

for row in response_top_referrer_websites_reference_month:
    top_referrer_websites_reference_month_rows.append({
        "referrer_website": row.get("label"),
        "referrer_url": row.get("url"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "month",
        "period_start": period_start,
        "period_end": period_end,
        "site_id": MATOMO_SITE_ID
    })

df_top_referrer_websites_reference_month = pd.DataFrame(
    top_referrer_websites_reference_month_rows
)

if not df_top_referrer_websites_reference_month.empty:
    df_top_referrer_websites_reference_month = (
        df_top_referrer_websites_reference_month
        .sort_values("visits_count", ascending=False)
    )

df_top_referrer_websites_reference_month["reference_month"] = REPORT_MONTH
df_top_referrer_websites_reference_month["environment"] = ENVIRONMENT
df_top_referrer_websites_reference_month["source"] = "matomo"
df_top_referrer_websites_reference_month["metric_definition"] = (
    "top external websites that referred visits to the tracked DSpace site, "
    "as classified by Matomo during the reference month"
)

output_top_referrer_websites_reference_month = (
    export_path("matomo_top_referrer_websites_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_top_referrer_websites_reference_month, 
    output_top_referrer_websites_reference_month,
    index=False
)

print(
    "Top referrer websites in the reference month extracted: "
    f"{len(df_top_referrer_websites_reference_month)}"
)

print(
    "Total visits from referrer websites in the reference month: "
    f"{df_top_referrer_websites_reference_month['visits_count'].sum() if not df_top_referrer_websites_reference_month.empty else 0}"
)

print(f"Reference period: {period_start} to {period_end}")

display(df_top_referrer_websites_reference_month.head(30))

print(f"File saved to: {output_top_referrer_websites_reference_month.resolve()}")

## **Pages and DSpace Sections**

### Top visited pages

In [ ]:
# =========================================================
# TOP VISITED PAGES
# =========================================================

# - page_title
# - page_label_path
# - page_url
# - pageviews_count
# - unique_pageviews_count
# - visits_count
# - average_time_on_page
# - bounce_rate
# - exit_rate
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

def normalize_matomo_response_to_list(response):
    """
    Normalizes Matomo API responses to a list of dictionaries.
    """

    if response is None:
        return []

    if isinstance(response, list):
        return response

    if isinstance(response, dict):
        if "value" in response and isinstance(response["value"], list):
            return response["value"]

        return [response]

    return []


def flatten_matomo_page_rows(rows, parent_label=None):
    """
    Flattens Matomo Actions.getPageUrls nested rows.

    Matomo can return page URLs as a hierarchical tree. This function keeps
    both top-level and nested page rows.
    """

    flattened_rows = []

    for row in rows:
        if not isinstance(row, dict):
            continue

        label = row.get("label")
        page_title = row.get("label")
        page_url = row.get("url")

        if parent_label:
            full_label = f"{parent_label} / {label}"
        else:
            full_label = label

        flattened_rows.append({
            "page_title": page_title,
            "page_label_path": full_label,
            "page_url": page_url,
            "pageviews_count": row.get("nb_hits", 0),
            "unique_pageviews_count": row.get("nb_uniq_pageviews", 0),
            "visits_count": row.get("nb_visits", 0),
            "average_time_on_page": row.get("avg_time_on_page"),
            "bounce_rate": row.get("bounce_rate"),
            "exit_rate": row.get("exit_rate")
        })

        subtable = row.get("subtable")
        if isinstance(subtable, list):
            flattened_rows.extend(
                flatten_matomo_page_rows(
                    subtable,
                    parent_label=full_label
                )
            )

    return flattened_rows


response_top_visited_pages = matomo_api_call(
    "Actions.getPageUrls",
    params={
        "period": "range",
        "date": "2000-01-01,today",
        "expanded": 1,
        "flat": 1,
        "filter_limit": -1
    }
)

page_rows = normalize_matomo_response_to_list(response_top_visited_pages)

flattened_page_rows = flatten_matomo_page_rows(page_rows)

df_top_visited_pages = pd.DataFrame(flattened_page_rows)

if not df_top_visited_pages.empty:
    df_top_visited_pages = (
        df_top_visited_pages
        .sort_values("pageviews_count", ascending=False)
    )

df_top_visited_pages["period"] = "range"
df_top_visited_pages["site_id"] = MATOMO_SITE_ID
df_top_visited_pages["reference_month"] = REPORT_MONTH
df_top_visited_pages["environment"] = ENVIRONMENT
df_top_visited_pages["source"] = "matomo"
df_top_visited_pages["metric_definition"] = (
    "top visited page URLs recorded by Matomo for the tracked DSpace site over the "
    "full available reporting range. Pageviews represent the number of times a page "
    "was viewed; visits represent the number of visits in which the page appeared."
)

output_top_visited_pages = export_path("matomo_top_visited_pages.csv")

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_top_visited_pages, output_top_visited_pages, index=False)

print(f"Visited pages extracted: {len(df_top_visited_pages)}")

print(
    "Total pageviews: "
    f"{df_top_visited_pages['pageviews_count'].sum() if not df_top_visited_pages.empty else 0}"
)

print(
    "Total unique pageviews: "
    f"{df_top_visited_pages['unique_pageviews_count'].sum() if not df_top_visited_pages.empty else 0}"
)

display(df_top_visited_pages.head(30))

print(f"File saved to: {output_top_visited_pages.resolve()}")

### Top visited pages in the reference month

In [ ]:
# =========================================================
# TOP VISITED PAGES IN THE REFERENCE MONTH
# =========================================================

# - page_title
# - page_label_path
# - page_url
# - pageviews_count
# - unique_pageviews_count
# - visits_count
# - average_time_on_page
# - bounce_rate
# - exit_rate
# - period
# - period_start
# - period_end
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")
report_month_date = reference_month_start.strftime("%Y-%m-%d")


def normalize_matomo_response_to_list(response):
    """
    Normalizes Matomo API responses to a list of dictionaries.
    """

    if response is None:
        return []

    if isinstance(response, list):
        return response

    if isinstance(response, dict):
        if "value" in response and isinstance(response["value"], list):
            return response["value"]

        return [response]

    return []


def flatten_matomo_page_rows(rows, parent_label=None):
    """
    Flattens Matomo Actions.getPageUrls nested rows.

    Matomo can return page URLs as a hierarchical tree. This function keeps
    both top-level and nested page rows.
    """

    flattened_rows = []

    for row in rows:
        if not isinstance(row, dict):
            continue

        label = row.get("label")
        page_title = row.get("label")
        page_url = row.get("url")

        if parent_label:
            full_label = f"{parent_label} / {label}"
        else:
            full_label = label

        flattened_rows.append({
            "page_title": page_title,
            "page_label_path": full_label,
            "page_url": page_url,
            "pageviews_count": row.get("nb_hits", 0),
            "unique_pageviews_count": row.get("nb_uniq_pageviews", 0),
            "visits_count": row.get("nb_visits", 0),
            "average_time_on_page": row.get("avg_time_on_page"),
            "bounce_rate": row.get("bounce_rate"),
            "exit_rate": row.get("exit_rate")
        })

        subtable = row.get("subtable")
        if isinstance(subtable, list):
            flattened_rows.extend(
                flatten_matomo_page_rows(
                    subtable,
                    parent_label=full_label
                )
            )

    return flattened_rows


response_top_visited_pages_reference_month = matomo_api_call(
    "Actions.getPageUrls",
    params={
        "period": "month",
        "date": report_month_date,
        "expanded": 1,
        "flat": 1,
        "filter_limit": -1
    }
)

page_rows = normalize_matomo_response_to_list(
    response_top_visited_pages_reference_month
)

flattened_page_rows = flatten_matomo_page_rows(page_rows)

df_top_visited_pages_reference_month = pd.DataFrame(flattened_page_rows)

if not df_top_visited_pages_reference_month.empty:
    df_top_visited_pages_reference_month = (
        df_top_visited_pages_reference_month
        .sort_values("pageviews_count", ascending=False)
    )

df_top_visited_pages_reference_month["period"] = "month"
df_top_visited_pages_reference_month["period_start"] = period_start
df_top_visited_pages_reference_month["period_end"] = period_end
df_top_visited_pages_reference_month["site_id"] = MATOMO_SITE_ID
df_top_visited_pages_reference_month["reference_month"] = REPORT_MONTH
df_top_visited_pages_reference_month["environment"] = ENVIRONMENT
df_top_visited_pages_reference_month["source"] = "matomo"
df_top_visited_pages_reference_month["metric_definition"] = (
    "top visited page URLs recorded by Matomo for the tracked DSpace site during "
    "the reference month. Pageviews represent the number of times a page was viewed; "
    "visits represent the number of visits in which the page appeared."
)

output_top_visited_pages_reference_month = (
    export_path("matomo_top_visited_pages_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_top_visited_pages_reference_month, 
    output_top_visited_pages_reference_month,
    index=False
)

print(
    "Visited pages in the reference month extracted: "
    f"{len(df_top_visited_pages_reference_month)}"
)

print(
    "Total pageviews in the reference month: "
    f"{df_top_visited_pages_reference_month['pageviews_count'].sum() if not df_top_visited_pages_reference_month.empty else 0}"
)

print(
    "Total unique pageviews in the reference month: "
    f"{df_top_visited_pages_reference_month['unique_pageviews_count'].sum() if not df_top_visited_pages_reference_month.empty else 0}"
)

print(f"Reference period: {period_start} to {period_end}")

display(df_top_visited_pages_reference_month.head(30))

print(f"File saved to: {output_top_visited_pages_reference_month.resolve()}")

### Typology of page views

In [ ]:
# =========================================================
# TYPOLOGY OF PAGE VIEWS
# =========================================================

# - page_typology
# - pageviews_count
# - unique_pageviews_count
# - visits_count
# - page_urls_count
# - page_urls
# - page_url_domain_filter
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

from urllib.parse import urlparse

PAGE_URL_DOMAIN_FILTER = config["matomo"].get(
    "domain_filter",
    "dspace-clarin-it.ilc.cnr.it"
)


def normalize_matomo_response_to_list(response):
    """
    Normalizes Matomo API responses to a list of dictionaries.
    """

    if response is None:
        return []

    if isinstance(response, list):
        return response

    if isinstance(response, dict):
        if "value" in response and isinstance(response["value"], list):
            return response["value"]

        return [response]

    return []


def flatten_matomo_page_rows(rows, parent_label=None):
    """
    Flattens Matomo Actions.getPageUrls nested rows.
    """

    flattened_rows = []

    for row in rows:
        if not isinstance(row, dict):
            continue

        label = row.get("label")
        page_url = row.get("url")

        if parent_label:
            full_label = f"{parent_label} / {label}"
        else:
            full_label = label

        flattened_rows.append({
            "page_title": label,
            "page_label_path": full_label,
            "page_url": page_url,
            "pageviews_count": row.get("nb_hits", 0),
            "unique_pageviews_count": row.get("nb_uniq_pageviews", 0),
            "visits_count": row.get("nb_visits", 0),
            "average_time_on_page": row.get("avg_time_on_page"),
            "bounce_rate": row.get("bounce_rate"),
            "exit_rate": row.get("exit_rate")
        })

        subtable = row.get("subtable")
        if isinstance(subtable, list):
            flattened_rows.extend(
                flatten_matomo_page_rows(
                    subtable,
                    parent_label=full_label
                )
            )

    return flattened_rows


def classify_dspace_page_typology(page_url):
    """
    Classifies DSpace page URLs into broad page typologies.

    Typologies:
    - homepage
    - search
    - item_page
    - collection_page
    - community_page
    - login_page
    - other
    """

    if page_url is None or str(page_url).strip() == "":
        return "unknown"

    parsed_url = urlparse(str(page_url).strip())
    path = parsed_url.path.rstrip("/")

    if path == "" or path == "/":
        return "homepage"

    if path == "/home":
        return "homepage"

    if path.startswith("/search"):
        return "search"

    if path.startswith("/items/"):
        return "item_page"

    if path.startswith("/collections/"):
        return "collection_page"

    if path.startswith("/communities/"):
        return "community_page"

    if path.startswith("/login"):
        return "login_page"

    return "other"


def join_limited_unique_urls(values, limit=50):
    """
    Joins up to N unique URLs for compact CSV output.
    """

    unique_values = sorted(
        set(
            str(value).strip()
            for value in values.dropna()
            if str(value).strip() != ""
        )
    )

    if len(unique_values) > limit:
        return (
            "; ".join(unique_values[:limit])
            + f"; ... [{len(unique_values) - limit} more]"
        )

    return "; ".join(unique_values)


response_pageviews = matomo_api_call(
    "Actions.getPageUrls",
    params={
        "period": "range",
        "date": "2000-01-01,today",
        "expanded": 1,
        "flat": 1,
        "filter_limit": -1
    }
)

page_rows = normalize_matomo_response_to_list(response_pageviews)
flattened_page_rows = flatten_matomo_page_rows(page_rows)

df_pageviews = pd.DataFrame(flattened_page_rows)

if not df_pageviews.empty:
    df_pageviews = df_pageviews[
        df_pageviews["page_url"]
        .fillna("")
        .str.contains(PAGE_URL_DOMAIN_FILTER, case=False, regex=False)
    ].copy()

if not df_pageviews.empty:
    df_pageviews["page_typology"] = df_pageviews["page_url"].apply(
        classify_dspace_page_typology
    )

    df_pageviews["pageviews_count"] = pd.to_numeric(
        df_pageviews["pageviews_count"],
        errors="coerce"
    ).fillna(0).astype(int)

    df_pageviews["unique_pageviews_count"] = pd.to_numeric(
        df_pageviews["unique_pageviews_count"],
        errors="coerce"
    ).fillna(0).astype(int)

    df_pageviews["visits_count"] = pd.to_numeric(
        df_pageviews["visits_count"],
        errors="coerce"
    ).fillna(0).astype(int)

    df_typology_of_page_views = (
        df_pageviews
        .groupby("page_typology", dropna=False)
        .agg(
            pageviews_count=("pageviews_count", "sum"),
            unique_pageviews_count=("unique_pageviews_count", "sum"),
            visits_count=("visits_count", "sum"),
            page_urls_count=("page_url", "nunique"),
            page_urls=("page_url", join_limited_unique_urls)
        )
        .reset_index()
        .sort_values("pageviews_count", ascending=False)
    )
else:
    df_typology_of_page_views = pd.DataFrame(
        columns=[
            "page_typology",
            "pageviews_count",
            "unique_pageviews_count",
            "visits_count",
            "page_urls_count",
            "page_urls"
        ]
    )

df_typology_of_page_views["page_url_domain_filter"] = PAGE_URL_DOMAIN_FILTER
df_typology_of_page_views["period"] = "range"
df_typology_of_page_views["site_id"] = MATOMO_SITE_ID
df_typology_of_page_views["reference_month"] = REPORT_MONTH
df_typology_of_page_views["environment"] = ENVIRONMENT
df_typology_of_page_views["source"] = "matomo"
df_typology_of_page_views["metric_definition"] = (
    "pageviews grouped by DSpace page typology. Typologies are derived from page URL "
    "patterns: homepage, search, item page, collection page, community page, login page, "
    "and other. Only page URLs containing the configured Matomo domain_filter are included."
)

output_typology_of_page_views = (
    export_path("matomo_typology_of_page_views.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_typology_of_page_views, 
    output_typology_of_page_views,
    index=False
)

print(f"Page typologies extracted: {len(df_typology_of_page_views)}")
print(f"Domain filter: {PAGE_URL_DOMAIN_FILTER}")
print(
    "Total pageviews by typology: "
    f"{df_typology_of_page_views['pageviews_count'].sum() if not df_typology_of_page_views.empty else 0}"
)

display(df_typology_of_page_views)

print(f"File saved to: {output_typology_of_page_views.resolve()}")

### Typology of page views on the reference month

In [ ]:
# =========================================================
# TYPOLOGY OF PAGE VIEWS IN THE REFERENCE MONTH
# =========================================================

# - page_typology
# - pageviews_count
# - unique_pageviews_count
# - visits_count
# - page_urls_count
# - page_urls
# - page_url_domain_filter
# - period
# - period_start
# - period_end
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

from urllib.parse import urlparse

PAGE_URL_DOMAIN_FILTER = config["matomo"].get(
    "domain_filter",
    "dspace-clarin-it.ilc.cnr.it"
)

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")
report_month_date = reference_month_start.strftime("%Y-%m-%d")


def normalize_matomo_response_to_list(response):
    """
    Normalizes Matomo API responses to a list of dictionaries.
    """

    if response is None:
        return []

    if isinstance(response, list):
        return response

    if isinstance(response, dict):
        if "value" in response and isinstance(response["value"], list):
            return response["value"]

        return [response]

    return []


def flatten_matomo_page_rows(rows, parent_label=None):
    """
    Flattens Matomo Actions.getPageUrls nested rows.
    """

    flattened_rows = []

    for row in rows:
        if not isinstance(row, dict):
            continue

        label = row.get("label")
        page_url = row.get("url")

        if parent_label:
            full_label = f"{parent_label} / {label}"
        else:
            full_label = label

        flattened_rows.append({
            "page_title": label,
            "page_label_path": full_label,
            "page_url": page_url,
            "pageviews_count": row.get("nb_hits", 0),
            "unique_pageviews_count": row.get("nb_uniq_pageviews", 0),
            "visits_count": row.get("nb_visits", 0),
            "average_time_on_page": row.get("avg_time_on_page"),
            "bounce_rate": row.get("bounce_rate"),
            "exit_rate": row.get("exit_rate")
        })

        subtable = row.get("subtable")
        if isinstance(subtable, list):
            flattened_rows.extend(
                flatten_matomo_page_rows(
                    subtable,
                    parent_label=full_label
                )
            )

    return flattened_rows


def classify_dspace_page_typology(page_url):
    """
    Classifies DSpace page URLs into broad page typologies.

    Typologies:
    - homepage
    - search
    - item_page
    - collection_page
    - community_page
    - login_page
    - other
    """

    if page_url is None or str(page_url).strip() == "":
        return "unknown"

    parsed_url = urlparse(str(page_url).strip())
    path = parsed_url.path.rstrip("/")

    if path == "" or path == "/":
        return "homepage"

    if path == "/home":
        return "homepage"

    if path.startswith("/search"):
        return "search"

    if path.startswith("/items/"):
        return "item_page"

    if path.startswith("/collections/"):
        return "collection_page"

    if path.startswith("/communities/"):
        return "community_page"

    if path.startswith("/login"):
        return "login_page"

    return "other"


def join_limited_unique_urls(values, limit=50):
    """
    Joins up to N unique URLs for compact CSV output.
    """

    unique_values = sorted(
        set(
            str(value).strip()
            for value in values.dropna()
            if str(value).strip() != ""
        )
    )

    if len(unique_values) > limit:
        return (
            "; ".join(unique_values[:limit])
            + f"; ... [{len(unique_values) - limit} more]"
        )

    return "; ".join(unique_values)


response_pageviews_reference_month = matomo_api_call(
    "Actions.getPageUrls",
    params={
        "period": "month",
        "date": report_month_date,
        "expanded": 1,
        "flat": 1,
        "filter_limit": -1
    }
)

page_rows = normalize_matomo_response_to_list(response_pageviews_reference_month)
flattened_page_rows = flatten_matomo_page_rows(page_rows)

df_pageviews_reference_month = pd.DataFrame(flattened_page_rows)

if not df_pageviews_reference_month.empty:
    df_pageviews_reference_month = df_pageviews_reference_month[
        df_pageviews_reference_month["page_url"]
        .fillna("")
        .str.contains(PAGE_URL_DOMAIN_FILTER, case=False, regex=False)
    ].copy()

if not df_pageviews_reference_month.empty:
    df_pageviews_reference_month["page_typology"] = (
        df_pageviews_reference_month["page_url"]
        .apply(classify_dspace_page_typology)
    )

    df_pageviews_reference_month["pageviews_count"] = pd.to_numeric(
        df_pageviews_reference_month["pageviews_count"],
        errors="coerce"
    ).fillna(0).astype(int)

    df_pageviews_reference_month["unique_pageviews_count"] = pd.to_numeric(
        df_pageviews_reference_month["unique_pageviews_count"],
        errors="coerce"
    ).fillna(0).astype(int)

    df_pageviews_reference_month["visits_count"] = pd.to_numeric(
        df_pageviews_reference_month["visits_count"],
        errors="coerce"
    ).fillna(0).astype(int)

    df_typology_of_page_views_reference_month = (
        df_pageviews_reference_month
        .groupby("page_typology", dropna=False)
        .agg(
            pageviews_count=("pageviews_count", "sum"),
            unique_pageviews_count=("unique_pageviews_count", "sum"),
            visits_count=("visits_count", "sum"),
            page_urls_count=("page_url", "nunique"),
            page_urls=("page_url", join_limited_unique_urls)
        )
        .reset_index()
        .sort_values("pageviews_count", ascending=False)
    )
else:
    df_typology_of_page_views_reference_month = pd.DataFrame(
        columns=[
            "page_typology",
            "pageviews_count",
            "unique_pageviews_count",
            "visits_count",
            "page_urls_count",
            "page_urls"
        ]
    )

df_typology_of_page_views_reference_month["page_url_domain_filter"] = (
    PAGE_URL_DOMAIN_FILTER
)
df_typology_of_page_views_reference_month["period"] = "month"
df_typology_of_page_views_reference_month["period_start"] = period_start
df_typology_of_page_views_reference_month["period_end"] = period_end
df_typology_of_page_views_reference_month["site_id"] = MATOMO_SITE_ID
df_typology_of_page_views_reference_month["reference_month"] = REPORT_MONTH
df_typology_of_page_views_reference_month["environment"] = ENVIRONMENT
df_typology_of_page_views_reference_month["source"] = "matomo"
df_typology_of_page_views_reference_month["metric_definition"] = (
    "pageviews grouped by DSpace page typology during the reference month. Typologies "
    "are derived from page URL patterns: homepage, search, item page, collection page, "
    "community page, login page, and other. Only page URLs containing the configured "
    "Matomo domain_filter are included."
)

output_typology_of_page_views_reference_month = (
    export_path("matomo_typology_of_page_views_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_typology_of_page_views_reference_month, 
    output_typology_of_page_views_reference_month,
    index=False
)

print(
    "Page typologies in the reference month extracted: "
    f"{len(df_typology_of_page_views_reference_month)}"
)

print(f"Domain filter: {PAGE_URL_DOMAIN_FILTER}")

print(
    "Total pageviews by typology in the reference month: "
    f"{df_typology_of_page_views_reference_month['pageviews_count'].sum() if not df_typology_of_page_views_reference_month.empty else 0}"
)

print(f"Reference period: {period_start} to {period_end}")

display(df_typology_of_page_views_reference_month)

print(f"File saved to: {output_typology_of_page_views_reference_month.resolve()}")

## **Devices and technology**

### Visits by device type

In [ ]:
# =========================================================
# VISITS BY DEVICE TYPE
# =========================================================

# - device_type
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_visits_by_device_type = matomo_api_call(
    "DevicesDetection.getType",
    params={
        "period": "range",
        "date": "2000-01-01,today",
        "filter_limit": -1
    }
)

visits_by_device_type_rows = []

for row in response_visits_by_device_type:
    visits_by_device_type_rows.append({
        "device_type": row.get("label"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "range",
        "site_id": MATOMO_SITE_ID
    })

df_visits_by_device_type = pd.DataFrame(visits_by_device_type_rows)

if not df_visits_by_device_type.empty:
    df_visits_by_device_type = (
        df_visits_by_device_type
        .sort_values("visits_count", ascending=False)
    )

df_visits_by_device_type["reference_month"] = REPORT_MONTH
df_visits_by_device_type["environment"] = ENVIRONMENT
df_visits_by_device_type["source"] = "matomo"
df_visits_by_device_type["metric_definition"] = (
    "visits grouped by device type recorded by Matomo for the tracked DSpace site "
    "over the full available reporting range"
)

output_visits_by_device_type = (
    export_path("matomo_visits_by_device_type.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_by_device_type, 
    output_visits_by_device_type,
    index=False
)

print(
    "Device types with visits: "
    f"{len(df_visits_by_device_type)}"
)

print(
    "Total visits by device type: "
    f"{df_visits_by_device_type['visits_count'].sum() if not df_visits_by_device_type.empty else 0}"
)

display(df_visits_by_device_type)

print(f"File saved to: {output_visits_by_device_type.resolve()}")

### Visits by device type in the reference month

In [ ]:
# =========================================================
# VISITS BY DEVICE TYPE IN THE REFERENCE MONTH
# =========================================================

# - device_type
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - period_start
# - period_end
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")
report_month_date = reference_month_start.strftime("%Y-%m-%d")

response_visits_by_device_type_reference_month = matomo_api_call(
    "DevicesDetection.getType",
    params={
        "period": "month",
        "date": report_month_date,
        "filter_limit": -1
    }
)

visits_by_device_type_reference_month_rows = []

for row in response_visits_by_device_type_reference_month:
    visits_by_device_type_reference_month_rows.append({
        "device_type": row.get("label"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "month",
        "period_start": period_start,
        "period_end": period_end,
        "site_id": MATOMO_SITE_ID
    })

df_visits_by_device_type_reference_month = pd.DataFrame(
    visits_by_device_type_reference_month_rows
)

if not df_visits_by_device_type_reference_month.empty:
    df_visits_by_device_type_reference_month = (
        df_visits_by_device_type_reference_month
        .sort_values("visits_count", ascending=False)
    )

df_visits_by_device_type_reference_month["reference_month"] = REPORT_MONTH
df_visits_by_device_type_reference_month["environment"] = ENVIRONMENT
df_visits_by_device_type_reference_month["source"] = "matomo"
df_visits_by_device_type_reference_month["metric_definition"] = (
    "visits grouped by device type recorded by Matomo for the tracked DSpace site "
    "during the reference month"
)

output_visits_by_device_type_reference_month = (
    export_path("matomo_visits_by_device_type_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_by_device_type_reference_month, 
    output_visits_by_device_type_reference_month,
    index=False
)

print(
    "Device types with visits in the reference month: "
    f"{len(df_visits_by_device_type_reference_month)}"
)

print(
    "Total visits by device type in the reference month: "
    f"{df_visits_by_device_type_reference_month['visits_count'].sum() if not df_visits_by_device_type_reference_month.empty else 0}"
)

print(f"Reference period: {period_start} to {period_end}")

display(df_visits_by_device_type_reference_month)

print(f"File saved to: {output_visits_by_device_type_reference_month.resolve()}")

### Visits by browser

In [ ]:
# =========================================================
# VISITS BY BROWSER
# =========================================================

# - browser
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_visits_by_browser = matomo_api_call(
    "DevicesDetection.getBrowsers",
    params={
        "period": "range",
        "date": "2000-01-01,today",
        "filter_limit": -1
    }
)

visits_by_browser_rows = []

for row in response_visits_by_browser:
    visits_by_browser_rows.append({
        "browser": row.get("label"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "range",
        "site_id": MATOMO_SITE_ID
    })

df_visits_by_browser = pd.DataFrame(visits_by_browser_rows)

if not df_visits_by_browser.empty:
    df_visits_by_browser = (
        df_visits_by_browser
        .sort_values("visits_count", ascending=False)
    )

df_visits_by_browser["reference_month"] = REPORT_MONTH
df_visits_by_browser["environment"] = ENVIRONMENT
df_visits_by_browser["source"] = "matomo"
df_visits_by_browser["metric_definition"] = (
    "visits grouped by browser recorded by Matomo for the tracked DSpace site "
    "over the full available reporting range"
)

output_visits_by_browser = (
    export_path("matomo_visits_by_browser.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_by_browser, 
    output_visits_by_browser,
    index=False
)

print(
    "Browsers with visits: "
    f"{len(df_visits_by_browser)}"
)

print(
    "Total visits by browser: "
    f"{df_visits_by_browser['visits_count'].sum() if not df_visits_by_browser.empty else 0}"
)

display(df_visits_by_browser.head(30))

print(f"File saved to: {output_visits_by_browser.resolve()}")

### Visits by browser in the reference month

In [ ]:
# =========================================================
# VISITS BY BROWSER IN THE REFERENCE MONTH
# =========================================================

# - browser
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - period_start
# - period_end
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")
report_month_date = reference_month_start.strftime("%Y-%m-%d")

response_visits_by_browser_reference_month = matomo_api_call(
    "DevicesDetection.getBrowsers",
    params={
        "period": "month",
        "date": report_month_date,
        "filter_limit": -1
    }
)

visits_by_browser_reference_month_rows = []

for row in response_visits_by_browser_reference_month:
    visits_by_browser_reference_month_rows.append({
        "browser": row.get("label"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "month",
        "period_start": period_start,
        "period_end": period_end,
        "site_id": MATOMO_SITE_ID
    })

df_visits_by_browser_reference_month = pd.DataFrame(
    visits_by_browser_reference_month_rows
)

if not df_visits_by_browser_reference_month.empty:
    df_visits_by_browser_reference_month = (
        df_visits_by_browser_reference_month
        .sort_values("visits_count", ascending=False)
    )

df_visits_by_browser_reference_month["reference_month"] = REPORT_MONTH
df_visits_by_browser_reference_month["environment"] = ENVIRONMENT
df_visits_by_browser_reference_month["source"] = "matomo"
df_visits_by_browser_reference_month["metric_definition"] = (
    "visits grouped by browser recorded by Matomo for the tracked DSpace site "
    "during the reference month"
)

output_visits_by_browser_reference_month = (
    export_path("matomo_visits_by_browser_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_by_browser_reference_month, 
    output_visits_by_browser_reference_month,
    index=False
)

print(
    "Browsers with visits in the reference month: "
    f"{len(df_visits_by_browser_reference_month)}"
)

print(
    "Total visits by browser in the reference month: "
    f"{df_visits_by_browser_reference_month['visits_count'].sum() if not df_visits_by_browser_reference_month.empty else 0}"
)

print(f"Reference period: {period_start} to {period_end}")

display(df_visits_by_browser_reference_month.head(30))

print(f"File saved to: {output_visits_by_browser_reference_month.resolve()}")

### Visits by operating system

In [ ]:
# =========================================================
# VISITS BY OPERATING SYSTEM
# =========================================================

# - operating_system
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

response_visits_by_operating_system = matomo_api_call(
    "DevicesDetection.getOsFamilies",
    params={
        "period": "range",
        "date": "2000-01-01,today",
        "filter_limit": -1
    }
)

visits_by_operating_system_rows = []

for row in response_visits_by_operating_system:
    visits_by_operating_system_rows.append({
        "operating_system": row.get("label"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "range",
        "site_id": MATOMO_SITE_ID
    })

df_visits_by_operating_system = pd.DataFrame(
    visits_by_operating_system_rows
)

if not df_visits_by_operating_system.empty:
    df_visits_by_operating_system = (
        df_visits_by_operating_system
        .sort_values("visits_count", ascending=False)
    )

df_visits_by_operating_system["reference_month"] = REPORT_MONTH
df_visits_by_operating_system["environment"] = ENVIRONMENT
df_visits_by_operating_system["source"] = "matomo"
df_visits_by_operating_system["metric_definition"] = (
    "visits grouped by operating system family recorded by Matomo for the tracked "
    "DSpace site over the full available reporting range"
)

output_visits_by_operating_system = (
    export_path("matomo_visits_by_operating_system.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_by_operating_system, 
    output_visits_by_operating_system,
    index=False
)

print(
    "Operating systems with visits: "
    f"{len(df_visits_by_operating_system)}"
)

print(
    "Total visits by operating system: "
    f"{df_visits_by_operating_system['visits_count'].sum() if not df_visits_by_operating_system.empty else 0}"
)

display(df_visits_by_operating_system.head(30))

print(f"File saved to: {output_visits_by_operating_system.resolve()}")

### Visits by operating system in the reference month

In [ ]:
# =========================================================
# VISITS BY OPERATING SYSTEM IN THE REFERENCE MONTH
# =========================================================

# - operating_system
# - visits_count
# - unique_visitors_count
# - actions_count
# - bounce_count
# - bounce_rate
# - actions_per_visit
# - average_time_on_site
# - period
# - period_start
# - period_end
# - site_id
# - reference_month
# - environment
# - source
# - metric_definition

reference_month_start = pd.to_datetime(f"{REPORT_MONTH}-01")
reference_month_end = reference_month_start + pd.DateOffset(months=1)

period_start = reference_month_start.strftime("%Y-%m-%d")
period_end = (reference_month_end - pd.DateOffset(days=1)).strftime("%Y-%m-%d")
report_month_date = reference_month_start.strftime("%Y-%m-%d")

response_visits_by_operating_system_reference_month = matomo_api_call(
    "DevicesDetection.getOsFamilies",
    params={
        "period": "month",
        "date": report_month_date,
        "filter_limit": -1
    }
)

visits_by_operating_system_reference_month_rows = []

for row in response_visits_by_operating_system_reference_month:
    visits_by_operating_system_reference_month_rows.append({
        "operating_system": row.get("label"),
        "visits_count": row.get("nb_visits", 0),
        "unique_visitors_count": row.get("nb_uniq_visitors", 0),
        "actions_count": row.get("nb_actions", 0),
        "bounce_count": row.get("bounce_count", 0),
        "bounce_rate": row.get("bounce_rate"),
        "actions_per_visit": row.get("nb_actions_per_visit"),
        "average_time_on_site": row.get("avg_time_on_site"),
        "period": "month",
        "period_start": period_start,
        "period_end": period_end,
        "site_id": MATOMO_SITE_ID
    })

df_visits_by_operating_system_reference_month = pd.DataFrame(
    visits_by_operating_system_reference_month_rows
)

if not df_visits_by_operating_system_reference_month.empty:
    df_visits_by_operating_system_reference_month = (
        df_visits_by_operating_system_reference_month
        .sort_values("visits_count", ascending=False)
    )

df_visits_by_operating_system_reference_month["reference_month"] = REPORT_MONTH
df_visits_by_operating_system_reference_month["environment"] = ENVIRONMENT
df_visits_by_operating_system_reference_month["source"] = "matomo"
df_visits_by_operating_system_reference_month["metric_definition"] = (
    "visits grouped by operating system family recorded by Matomo for the tracked "
    "DSpace site during the reference month"
)

output_visits_by_operating_system_reference_month = (
    export_path("matomo_visits_by_operating_system_reference_month.csv")
)

export_dir.mkdir(parents=True, exist_ok=True)
write_csv(df_visits_by_operating_system_reference_month, 
    output_visits_by_operating_system_reference_month,
    index=False
)

print(
    "Operating systems with visits in the reference month: "
    f"{len(df_visits_by_operating_system_reference_month)}"
)

print(
    "Total visits by operating system in the reference month: "
    f"{df_visits_by_operating_system_reference_month['visits_count'].sum() if not df_visits_by_operating_system_reference_month.empty else 0}"
)

print(f"Reference period: {period_start} to {period_end}")

display(df_visits_by_operating_system_reference_month.head(30))

print(f"File saved to: {output_visits_by_operating_system_reference_month.resolve()}")